In [1]:
# ============================================================
# EL CAPO 3.0
# INTERAKTIV REGISTRERING, MASSEIMPORT, DIAGRAM,
# KARAKTER OG TILBAKEMELDING
#
# Python 3.14
# Jupyter Notebook / JupyterLab
#
# HOVEDFUNKSJONER:
# - Valgfritt antall oppgaver
# - Tema, oppgavenummer og makspoeng
# - Manuell registrering av én elev
# - Innliming av én elevs poeng
# - Masseimport av hele klassen fra Excel
# - Første Excel-kolonne: elevnavn
# - Resten av Excel-kolonnene: poeng
# - Kontroll av antall poeng
# - Kontroll av makspoeng
# - Automatisk prosent og karakter
# - Automatisk elevtilbakemelding
# - Ett diagram per elev
# - Lagre valgt diagram som PNG eller PDF
# - Lagre alle diagrammer som PNG
# - Lagre og laste prøve/elevdata som JSON
# ============================================================


# ============================================================
# 1. IMPORTER PAKKER
# ============================================================

from pathlib import Path
import html
import json
import re
import textwrap

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch

import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# 2. DATA OG FILER
# ============================================================

proveoppsett = []

# Format:
#
# {
#     "Elevnavn": {
#         "poeng": [4, 2, 6],
#         "kommentar": ""
#     }
# }

elevdata = {}

poengfelter = []

RESULTATMAPPE = Path("Elevdiagrammer")

DATAFIL = Path("elevresultater_el_capo_3.json")


# ============================================================
# 3. INNSTILLINGER FOR TILBAKEMELDING
# ============================================================

GRENSE_OVING = 60
GRENSE_STERK = 80

MAKS_OVINGSOMRADER = 3
MAKS_STERKE_OMRADER = 2


# ============================================================
# 4. FARGER
# ============================================================

FARGE_LAV = "#D9534F"
FARGE_MIDDELS = "#F0AD4E"
FARGE_GOD = "#5B9BD5"
FARGE_MEGET_GOD = "#2E8B57"

FARGE_TOTAL = "#7030A0"
FARGE_BAKGRUNN = "#FAFAFA"
FARGE_RUTENETT = "#D9D9D9"

FARGE_TILBAKEMELDING = "#F3F7FB"
FARGE_TILBAKEMELDING_KANT = "#AFC4D8"


# ============================================================
# 5. HJELPEFUNKSJONER
# ============================================================

def norsk_tall(verdi, desimaler=1):
    """
    Formaterer tall med norsk desimalkomma.
    """
    return f"{float(verdi):.{desimaler}f}".replace(".", ",")


def poengtall(verdi):
    """
    Viser hele tall uten desimal og desimaltall med komma.
    """
    verdi = float(verdi)

    if verdi.is_integer():
        return str(int(verdi))

    tekst = f"{verdi:.10f}".rstrip("0").rstrip(".")
    return tekst.replace(".", ",")


def sikkert_filnavn(tekst):
    """
    Lager et filnavn som fungerer i Windows.
    """
    tekst = str(tekst).strip()

    tekst = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        tekst
    )

    tekst = re.sub(
        r"\s+",
        "_",
        tekst
    )

    return tekst.strip("._") or "elev"


def velg_farge(prosent):
    """
    Velger søylefarge etter prosentresultatet.
    """
    if prosent < 40:
        return FARGE_LAV

    if prosent < 60:
        return FARGE_MIDDELS

    if prosent < 80:
        return FARGE_GOD

    return FARGE_MEGET_GOD


def formater_oppramsing(navn):
    """
    Lager en naturlig norsk oppramsing.
    """
    navn = list(navn)

    if not navn:
        return ""

    if len(navn) == 1:
        return navn[0]

    if len(navn) == 2:
        return f"{navn[0]} og {navn[1]}"

    return (
        f"{', '.join(navn[:-1])} "
        f"og {navn[-1]}"
    )


def finn_elev(navn):
    """
    Finner elev uten å skille mellom store og små bokstaver.
    """
    sokenavn = str(navn).strip().casefold()

    for eksisterende_navn in elevdata:
        if eksisterende_navn.casefold() == sokenavn:
            return eksisterende_navn

    return None


def visningsnavn(element):
    """
    Lager etikett til poengfelt og diagram.
    """
    return (
        f"{element['oppgave']}: "
        f"{element['tema']}"
    )


def vis_status(melding, meldingstype="info"):
    """
    Viser en statusmelding i programvinduet.
    """
    stiler = {
        "info": {
            "bakgrunn": "#F3F4F6",
            "kant": "#6B7280",
        },
        "suksess": {
            "bakgrunn": "#ECF8EF",
            "kant": "#2E8B57",
        },
        "advarsel": {
            "bakgrunn": "#FFF4E5",
            "kant": "#F0AD4E",
        },
        "feil": {
            "bakgrunn": "#FFF1F1",
            "kant": "#D9534F",
        },
    }

    stil = stiler.get(
        meldingstype,
        stiler["info"]
    )

    sikker_melding = html.escape(
        str(melding)
    ).replace(
        "\n",
        "<br>"
    )

    statusfelt.value = (
        f"<div style='"
        f"padding:10px 14px;"
        f"margin-top:8px;"
        f"background:{stil['bakgrunn']};"
        f"border-left:5px solid {stil['kant']};"
        f"border-radius:5px;"
        f"color:#303030;"
        f"'>"
        f"{sikker_melding}"
        f"</div>"
    )


def les_poengtall(tekst):
    """
    Leser norsk eller internasjonalt desimaltall.

    Eksempler:
        4
        4,00
        1,50
        2.5
    """
    if isinstance(tekst, (int, float, np.number)):
        verdi = float(tekst)

    else:
        tekst = str(tekst).strip()

        tekst = (
            tekst
            .replace("\u00A0", "")
            .replace(" ", "")
            .replace(",", ".")
        )

        if not tekst:
            raise ValueError(
                "Poengcellen er tom."
            )

        try:
            verdi = float(tekst)

        except ValueError:
            raise ValueError(
                f"Kunne ikke tolke {tekst!r} som poeng."
            )

    if not np.isfinite(verdi):
        raise ValueError(
            "Poengverdien er ikke et gyldig tall."
        )

    return verdi


# ============================================================
# 6. KARAKTERBEREGNING
# ============================================================

def karakter_fra_prosent(prosent):
    """
    Regner ut karakter fra samlet prosent.

    Prosenten avrundes til nærmeste hele prosent før
    karakteren bestemmes.
    """
    avrundet_prosent = int(
        np.floor(float(prosent) + 0.5)
    )

    avrundet_prosent = max(
        0,
        min(100, avrundet_prosent)
    )

    karakterskala = [
        (95, "6"),
        (92, "6-"),
        (88, "5+"),
        (84, "5"),
        (80, "5-"),
        (75, "4+"),
        (65, "4"),
        (60, "4-"),
        (55, "3+"),
        (45, "3"),
        (40, "3-"),
        (35, "2+"),
        (29, "2"),
        (21, "2-"),
        (0, "1"),
    ]

    for nedre_grense, karakter in karakterskala:
        if avrundet_prosent >= nedre_grense:
            return karakter, avrundet_prosent

    return "1", avrundet_prosent


# ============================================================
# 7. LES PRØVEOPPSETT
# ============================================================

def les_proveoppsett():
    """
    Leser prøveoppsettet.

    Format:
        Tema;Oppgavenummer;Makspoeng
    """
    linjer = [
        linje.strip()
        for linje in oppsett_felt.value.splitlines()
        if linje.strip()
    ]

    if not linjer:
        raise ValueError(
            "Skriv minst én oppgave."
        )

    nytt_oppsett = []

    for linjenummer, linje in enumerate(
        linjer,
        start=1
    ):
        deler = [
            deltekst.strip()
            for deltekst in linje.split(";")
        ]

        if len(deler) != 3:
            raise ValueError(
                f"Linje {linjenummer} må ha formatet "
                "Tema;Oppgavenummer;Makspoeng."
            )

        tema, oppgavenummer, maks_tekst = deler

        if not tema:
            raise ValueError(
                f"Linje {linjenummer} mangler tema."
            )

        if not oppgavenummer:
            raise ValueError(
                f"Linje {linjenummer} mangler oppgavenummer."
            )

        try:
            makspoeng = les_poengtall(
                maks_tekst
            )

        except ValueError:
            raise ValueError(
                f"Ugyldig makspoeng på linje "
                f"{linjenummer}: {maks_tekst!r}."
            )

        if makspoeng <= 0:
            raise ValueError(
                f"Makspoeng på linje {linjenummer} "
                "må være større enn 0."
            )

        nytt_oppsett.append(
            {
                "tema": tema,
                "oppgave": oppgavenummer,
                "maks": makspoeng,
            }
        )

    etiketter = [
        visningsnavn(element).casefold()
        for element in nytt_oppsett
    ]

    if len(set(etiketter)) != len(etiketter):
        raise ValueError(
            "Hver kombinasjon av oppgavenummer og tema "
            "må være unik."
        )

    return nytt_oppsett


# ============================================================
# 8. POENGFELT
# ============================================================

def lag_poengfelt(element):
    """
    Lager ett manuelt poengfelt.
    """
    etikett = visningsnavn(
        element
    )

    maksimum = element["maks"]

    return widgets.FloatText(
        value=0.0,
        step=0.5,
        description=f"{etikett}:",
        style={
            "description_width": "250px"
        },
        layout=widgets.Layout(
            width="420px"
        ),
        tooltip=(
            "Maksimal poengsum: "
            f"{poengtall(maksimum)}"
        )
    )


# ============================================================
# 9. OPPRETT PRØVE
# ============================================================

def opprett_prove(knapp=None):
    """
    Oppretter prøve og poengfelt.
    """
    global proveoppsett
    global poengfelter

    try:
        nytt_oppsett = les_proveoppsett()

        if elevdata:
            raise ValueError(
                "Slett alle registrerte elever før "
                "prøveoppsettet endres."
            )

        proveoppsett = nytt_oppsett

        poengfelter = [
            lag_poengfelt(element)
            for element in proveoppsett
        ]

        poeng_rutenett.children = tuple(
            poengfelter
        )

        totalt_makspoeng = sum(
            element["maks"]
            for element in proveoppsett
        )

        proveinfo.value = (
            "<div style='"
            "padding:12px;"
            "margin:10px 0;"
            "background:#EEF5FF;"
            "border-left:5px solid #5B9BD5;"
            "border-radius:5px;"
            "'>"
            "<b>Aktiv prøve:</b> "
            f"{len(proveoppsett)} oppgaver, "
            f"totalt <b>{poengtall(totalt_makspoeng)} "
            "poeng</b>."
            "</div>"
        )

        registreringsdel.layout.display = ""

        tom_feltene(
            vis_melding=False
        )

        vis_status(
            "Prøveoppsettet er opprettet. "
            "Du kan nå lime inn hele klassen fra Excel.",
            "suksess"
        )

        with diagramomrade:
            clear_output(wait=True)

            display(
                widgets.HTML(
                    value=(
                        "<div style='"
                        "padding:25px;"
                        "text-align:center;"
                        "color:#6B7280;"
                        "border:1px dashed #BFC5CE;"
                        "border-radius:6px;"
                        "'>"
                        "Prøven er klar. Lim inn hele "
                        "klassen fra Excel eller registrer "
                        "én elev."
                        "</div>"
                    )
                )
            )

    except Exception as feil:
        vis_status(
            f"Kunne ikke opprette prøve: {feil}",
            "feil"
        )


# ============================================================
# 10. LES ÉN ELEVS INNLIMTE POENG
# ============================================================

def les_innlimte_poeng():
    """
    Leser poeng for én elev.

    Godtar:
    - én verdi per linje
    - én Excel-rad
    - tabulator
    - semikolon
    - norsk desimalkomma
    """
    tekst = innlimingsfelt.value.strip()

    if not tekst:
        raise ValueError(
            "Lim inn elevens poeng først."
        )

    tekst = tekst.replace("\t", "\n")
    tekst = tekst.replace(";", "\n")

    verdier = []

    for deltekst in tekst.splitlines():
        deltekst = deltekst.strip()

        if not deltekst:
            continue

        verdier.append(
            les_poengtall(deltekst)
        )

    return verdier


def kontroller_poengliste(
    elevnavn,
    poengliste
):
    """
    Kontrollerer antall poeng og makspoeng.
    """
    if len(poengliste) != len(proveoppsett):
        raise ValueError(
            f"{elevnavn}: Fant {len(poengliste)} "
            f"poengverdier, men prøven har "
            f"{len(proveoppsett)} oppgaver."
        )

    for element, oppnådd in zip(
        proveoppsett,
        poengliste
    ):
        etikett = visningsnavn(
            element
        )

        maksimum = float(
            element["maks"]
        )

        if not np.isfinite(oppnådd):
            raise ValueError(
                f"{elevnavn}, {etikett}: "
                "Poengsummen er ikke et gyldig tall."
            )

        if oppnådd < 0:
            raise ValueError(
                f"{elevnavn}, {etikett}: "
                "Poengsummen kan ikke være negativ."
            )

        if oppnådd > maksimum:
            raise ValueError(
                f"{elevnavn}, {etikett}: "
                f"Eleven har {poengtall(oppnådd)} poeng, "
                f"men makspoeng er "
                f"{poengtall(maksimum)}."
            )


def fordel_innlimte_poeng(knapp=None):
    """
    Fordeler én elevs innlimte poeng til feltene.
    """
    try:
        if not proveoppsett:
            raise ValueError(
                "Opprett prøveoppsettet først."
            )

        verdier = les_innlimte_poeng()

        kontroller_poengliste(
            navnefelt.value.strip() or "Eleven",
            verdier
        )

        for felt, verdi in zip(
            poengfelter,
            verdier
        ):
            felt.value = verdi

        samlet_poeng = sum(
            verdier
        )

        samlet_maksimum = sum(
            element["maks"]
            for element in proveoppsett
        )

        vis_status(
            f"Poengene er fordelt. Foreløpig poengsum: "
            f"{poengtall(samlet_poeng)} av "
            f"{poengtall(samlet_maksimum)}.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke fordele poengene: {feil}",
            "feil"
        )


# ============================================================
# 11. MASSEIMPORT AV HELE KLASSEN FRA EXCEL
# ============================================================

def les_excelrader():
    """
    Leser hele Excel-området.

    Forventet oppsett:
        Kolonne 1: Elevnavn
        Kolonne 2: Poeng oppgave 1
        Kolonne 3: Poeng oppgave 2
        ...
    """
    if not proveoppsett:
        raise ValueError(
            "Opprett prøveoppsettet først."
        )

    tekst = masseinnlimingsfelt.value.strip()

    if not tekst:
        raise ValueError(
            "Lim inn Excel-området først."
        )

    forventet_antall = len(proveoppsett) + 1
    elever = []

    linjer = tekst.splitlines()

    for linjenummer, linje in enumerate(
        linjer,
        start=1
    ):
        if not linje.strip():
            continue

        # Excel bruker tabulator mellom kolonnene.
        if "\t" in linje:
            kolonner = linje.split("\t")

        # Semikolon støttes som reserveformat.
        elif ";" in linje:
            kolonner = linje.split(";")

        else:
            raise ValueError(
                f"Rad {linjenummer} inneholder ikke "
                "Excel-kolonner. Marker hele Excel-området "
                "og kopier det samlet."
            )

        kolonner = [
            kolonne.strip()
            for kolonne in kolonner
        ]

        # Fjern helt tomme kolonner på slutten.
        while kolonner and not kolonner[-1]:
            kolonner.pop()

        if not kolonner:
            continue

        forste_verdi = kolonner[0].strip()

        # Hopp over eventuell overskriftsrad.
        if forste_verdi.casefold() in {
            "navn",
            "elev",
            "elevnavn",
            "student",
            "studentnavn",
        }:
            continue

        if len(kolonner) != forventet_antall:
            raise ValueError(
                f"Rad {linjenummer} har "
                f"{len(kolonner)} kolonner. Det skal være "
                f"{forventet_antall}: én navnekolonne og "
                f"{len(proveoppsett)} poengkolonner."
            )

        elevnavn = kolonner[0].strip()

        if not elevnavn:
            raise ValueError(
                f"Rad {linjenummer} mangler elevnavn."
            )

        poengliste = []

        for oppgavenummer, poengtekst in enumerate(
            kolonner[1:],
            start=1
        ):
            try:
                poeng = les_poengtall(
                    poengtekst
                )

            except ValueError as feil:
                raise ValueError(
                    f"Rad {linjenummer}, {elevnavn}, "
                    f"poengkolonne {oppgavenummer}: {feil}"
                )

            poengliste.append(
                poeng
            )

        elever.append(
            {
                "navn": elevnavn,
                "poeng": poengliste,
            }
        )

    if not elever:
        raise ValueError(
            "Fant ingen elever i Excel-området."
        )

    return elever


def kontroller_excelimport(elever):
    """
    Kontrollerer hele importen før noen elever registreres.
    """
    navn_i_import = set()

    for elev in elever:
        elevnavn = elev["navn"]
        sammenligningsnavn = elevnavn.casefold()

        if sammenligningsnavn in navn_i_import:
            raise ValueError(
                f"{elevnavn} forekommer mer enn én gang "
                "i Excel-området."
            )

        navn_i_import.add(
            sammenligningsnavn
        )

        kontroller_poengliste(
            elevnavn,
            elev["poeng"]
        )


def forhåndsvis_excelimport(knapp=None):
    """
    Kontrollerer innholdet uten å registrere elevene.
    """
    try:
        elever = les_excelrader()

        kontroller_excelimport(
            elever
        )

        samlet_maksimum = sum(
            element["maks"]
            for element in proveoppsett
        )

        with meldingomrade:
            clear_output(wait=True)

            print(
                f"Excel-området inneholder "
                f"{len(elever)} elever."
            )
            print()

            for elev in elever:
                samlet_poeng = sum(
                    elev["poeng"]
                )

                prosent = (
                    100
                    * samlet_poeng
                    / samlet_maksimum
                )

                karakter, avrundet = (
                    karakter_fra_prosent(
                        prosent
                    )
                )

                print(
                    f"{elev['navn']}: "
                    f"{poengtall(samlet_poeng)} av "
                    f"{poengtall(samlet_maksimum)} poeng, "
                    f"{avrundet} %, karakter {karakter}"
                )

        vis_status(
            f"Kontrollen var vellykket. "
            f"{len(elever)} elever er klare til import.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Excel-kontrollen fant en feil: {feil}",
            "feil"
        )


def importer_hele_klassen(knapp=None):
    """
    Importerer eller oppdaterer hele klassen.

    Eksisterende elever med samme navn oppdateres.
    Nye elever legges til.
    """
    try:
        elever = les_excelrader()

        # Hele tabellen kontrolleres før elevdata endres.
        kontroller_excelimport(
            elever
        )

        antall_nye = 0
        antall_oppdaterte = 0
        importerte_navn = []

        for elev in elever:
            elevnavn = elev["navn"]

            eksisterende_navn = finn_elev(
                elevnavn
            )

            if eksisterende_navn is None:
                lagringsnavn = elevnavn
                antall_nye += 1

            else:
                lagringsnavn = eksisterende_navn
                antall_oppdaterte += 1

            tidligere_kommentar = (
                elevdata
                .get(lagringsnavn, {})
                .get("kommentar", "")
            )

            elevdata[lagringsnavn] = {
                "poeng": [
                    float(verdi)
                    for verdi in elev["poeng"]
                ],
                "kommentar": tidligere_kommentar,
            }

            importerte_navn.append(
                lagringsnavn
            )

        oppdater_elevmeny(
            valgt_navn=importerte_navn[0]
        )

        last_elev_i_feltene(
            importerte_navn[0]
        )

        vis_valgt_diagram()

        with meldingomrade:
            clear_output(wait=True)

            print("IMPORT FULLFØRT")
            print("=" * 50)
            print(
                f"Antall leste elever: "
                f"{len(importerte_navn)}"
            )
            print(
                f"Nye elever: {antall_nye}"
            )
            print(
                f"Oppdaterte elever: "
                f"{antall_oppdaterte}"
            )
            print()
            print(
                "Elevene er registrert. Bruk "
                "'Lagre alle PNG' dersom du vil lagre "
                "ett diagram per elev."
            )

        vis_status(
            f"Importen er fullført. "
            f"{antall_nye} elever ble lagt til, og "
            f"{antall_oppdaterte} elever ble oppdatert.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke importere Excel-området: {feil}",
            "feil"
        )


def importer_og_lagre_alle(knapp=None):
    """
    Importerer hele klassen og lagrer deretter
    ett PNG-diagram per elev.
    """
    try:
        elever = les_excelrader()

        kontroller_excelimport(
            elever
        )

        importerte_navn = []
        antall_nye = 0
        antall_oppdaterte = 0

        for elev in elever:
            elevnavn = elev["navn"]

            eksisterende_navn = finn_elev(
                elevnavn
            )

            if eksisterende_navn is None:
                lagringsnavn = elevnavn
                antall_nye += 1

            else:
                lagringsnavn = eksisterende_navn
                antall_oppdaterte += 1

            tidligere_kommentar = (
                elevdata
                .get(lagringsnavn, {})
                .get("kommentar", "")
            )

            elevdata[lagringsnavn] = {
                "poeng": [
                    float(verdi)
                    for verdi in elev["poeng"]
                ],
                "kommentar": tidligere_kommentar,
            }

            importerte_navn.append(
                lagringsnavn
            )

        oppdater_elevmeny(
            valgt_navn=importerte_navn[0]
        )

        RESULTATMAPPE.mkdir(
            parents=True,
            exist_ok=True
        )

        with meldingomrade:
            clear_output(wait=True)

            print(
                f"Importerte {len(importerte_navn)} elever."
            )
            print()
            print("Lager diagrammer ...")
            print()

            for nummer, elevnavn in enumerate(
                sorted(
                    importerte_navn,
                    key=str.casefold
                ),
                start=1
            ):
                fil = lag_diagram(
                    elevnavn,
                    lagre=True,
                    filformat="png",
                    vis=False
                )

                print(
                    f"{nummer} av {len(importerte_navn)}: "
                    f"{fil.name}"
                )

            print()
            print("Alle diagrammene er lagret i:")
            print(RESULTATMAPPE.resolve())

        last_elev_i_feltene(
            importerte_navn[0]
        )

        vis_valgt_diagram()

        vis_status(
            f"Import og lagring er fullført. "
            f"{len(importerte_navn)} diagrammer ble lagret "
            f"i {RESULTATMAPPE.resolve()}",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke importere og lagre: {feil}",
            "feil"
        )


# ============================================================
# 12. HENT MANUELL INNTASTING
# ============================================================

def hent_inntasting():
    """
    Henter og kontrollerer én elevs manuelle inntasting.
    """
    if not proveoppsett:
        raise ValueError(
            "Opprett prøveoppsettet først."
        )

    elevnavn = navnefelt.value.strip()

    if not elevnavn:
        raise ValueError(
            "Skriv elevens navn."
        )

    poengliste = [
        float(felt.value)
        for felt in poengfelter
    ]

    kontroller_poengliste(
        elevnavn,
        poengliste
    )

    kommentar = kommentarfelt.value.strip()

    return (
        elevnavn,
        poengliste,
        kommentar
    )


# ============================================================
# 13. BEREGN RESULTAT
# ============================================================

def beregn_resultat(elevnavn):
    """
    Beregner oppgaveprosenter, totalprosent og karakter.
    """
    if elevnavn not in elevdata:
        raise KeyError(
            f"Fant ikke eleven {elevnavn!r}."
        )

    poengliste = elevdata[elevnavn]["poeng"]

    resultater = []

    for element, oppnådd in zip(
        proveoppsett,
        poengliste
    ):
        maksimum = float(
            element["maks"]
        )

        prosent = (
            100
            * float(oppnådd)
            / maksimum
        )

        resultater.append(
            {
                "tema": element["tema"],
                "oppgave": element["oppgave"],
                "etikett": visningsnavn(element),
                "oppnådd": float(oppnådd),
                "maksimum": maksimum,
                "prosent": prosent,
            }
        )

    samlet_poeng = sum(
        poengliste
    )

    samlet_maksimum = sum(
        element["maks"]
        for element in proveoppsett
    )

    samlet_prosent = (
        100
        * samlet_poeng
        / samlet_maksimum
    )

    karakter, avrundet_prosent = (
        karakter_fra_prosent(
            samlet_prosent
        )
    )

    return (
        resultater,
        samlet_poeng,
        samlet_maksimum,
        samlet_prosent,
        avrundet_prosent,
        karakter
    )


# ============================================================
# 14. AUTOMATISK TILBAKEMELDING
# ============================================================

def lag_tilbakemelding(
    resultater,
    samlet_prosent,
    karakter,
    kommentar=""
):
    """
    Lager en positiv og konkret elevtilbakemelding.
    """
    ovingsomrader = sorted(
        [
            resultat
            for resultat in resultater
            if resultat["prosent"] < GRENSE_OVING
        ],
        key=lambda resultat: (
            resultat["prosent"],
            resultat["etikett"]
        )
    )[:MAKS_OVINGSOMRADER]

    sterke_omrader = sorted(
        [
            resultat
            for resultat in resultater
            if resultat["prosent"] >= GRENSE_STERK
        ],
        key=lambda resultat: (
            -resultat["prosent"],
            resultat["etikett"]
        )
    )[:MAKS_STERKE_OMRADER]

    tekstdeler = [
        f"Samlet resultat gir karakter {karakter}."
    ]

    if samlet_prosent < 40:
        tekstdeler.append(
            "Du bør arbeide videre med flere deler av "
            "prøven. Ta ett område om gangen og øv på "
            "framgangsmåten før du prøver tilsvarende "
            "oppgaver på egen hånd."
        )

    elif samlet_prosent < 60:
        tekstdeler.append(
            "Du mestrer deler av prøven, men noen temaer "
            "eller oppgavetyper trenger mer øving."
        )

    elif samlet_prosent < 80:
        tekstdeler.append(
            "Du viser god forståelse på mange deler av "
            "prøven. Videre øving på noen utvalgte områder "
            "kan gjøre resultatet enda bedre."
        )

    else:
        tekstdeler.append(
            "Du viser svært god forståelse av innholdet "
            "i prøven. Fortsett å bruke de gode "
            "framgangsmåtene dine."
        )

    if ovingsomrader:
        ovingsnavn = [
            resultat["etikett"]
            for resultat in ovingsomrader
        ]

        tekstdeler.append(
            "Du bør særlig øve mer på "
            f"{formater_oppramsing(ovingsnavn)}."
        )

    if sterke_omrader:
        sterke_navn = [
            resultat["etikett"]
            for resultat in sterke_omrader
        ]

        tekstdeler.append(
            "Du har vist særlig god mestring på "
            f"{formater_oppramsing(sterke_navn)}."
        )

    kommentar = kommentar.strip()

    if kommentar:
        tekstdeler.append(
            f"Lærerens kommentar: {kommentar}"
        )

    return " ".join(
        tekstdeler
    )


# ============================================================
# 15. LAG DIAGRAM
# ============================================================

def lag_diagram(
    elevnavn,
    lagre=False,
    filformat="png",
    vis=True
):
    """
    Lager søylediagram, karakter og tilbakemelding.
    """
    (
        resultater,
        samlet_poeng,
        samlet_maksimum,
        samlet_prosent,
        avrundet_prosent,
        karakter
    ) = beregn_resultat(
        elevnavn
    )

    kommentar = elevdata[elevnavn].get(
        "kommentar",
        ""
    )

    tilbakemelding = lag_tilbakemelding(
        resultater,
        samlet_prosent,
        karakter,
        kommentar
    )

    tilbakemelding_brytet = textwrap.fill(
        tilbakemelding,
        width=145
    )

    etiketter = [
        resultat["etikett"]
        for resultat in resultater
    ]

    prosenter = np.array(
        [
            resultat["prosent"]
            for resultat in resultater
        ],
        dtype=float
    )

    figurbredde = max(
        14,
        len(etiketter) * 1.15
    )

    fig, ax = plt.subplots(
        figsize=(
            figurbredde,
            9.8
        )
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.98,
        top=0.88,
        bottom=0.38
    )

    fig.patch.set_facecolor(
        "white"
    )

    ax.set_facecolor(
        FARGE_BAKGRUNN
    )

    x = np.arange(
        len(etiketter)
    )

    soyler = ax.bar(
        x,
        prosenter,
        width=0.70,
        color=[
            velg_farge(prosent)
            for prosent in prosenter
        ],
        edgecolor="white",
        linewidth=1.2,
        zorder=3
    )

    ax.set_title(
        f"Resultat for: {elevnavn}",
        fontsize=18,
        fontweight="bold",
        color="#4F4F4F",
        pad=18
    )

    ax.set_xlabel(
        "Tema / oppgave",
        fontsize=12,
        fontweight="bold",
        labelpad=14
    )

    ax.set_ylabel(
        "Prosent riktig",
        fontsize=12,
        fontweight="bold"
    )

    ax.set_ylim(
        0,
        110
    )

    ax.set_yticks(
        np.arange(
            0,
            101,
            10
        )
    )

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter(
            xmax=100,
            decimals=0
        )
    )

    ax.set_xticks(x)

    lengste_navn = max(
        len(navn)
        for navn in etiketter
    )

    if (
        len(etiketter) <= 7
        and lengste_navn <= 18
    ):
        rotasjon = 15
        skriftstorrelse = 10

    elif len(etiketter) <= 12:
        rotasjon = 30
        skriftstorrelse = 9

    else:
        rotasjon = 45
        skriftstorrelse = 8

    ax.set_xticklabels(
        etiketter,
        rotation=rotasjon,
        ha="right",
        fontsize=skriftstorrelse
    )

    ax.grid(
        axis="y",
        color=FARGE_RUTENETT,
        linewidth=1,
        alpha=0.9,
        zorder=0
    )

    ax.grid(
        axis="x",
        visible=False
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color(
        "#BFBFBF"
    )

    ax.spines["bottom"].set_color(
        "#BFBFBF"
    )

    for soyle, resultat in zip(
        soyler,
        resultater
    ):
        prosent = resultat["prosent"]
        oppnådd = resultat["oppnådd"]
        maksimum = resultat["maksimum"]

        senter = (
            soyle.get_x()
            + soyle.get_width() / 2
        )

        ax.text(
            senter,
            prosent + 1.5,
            f"{norsk_tall(prosent)} %",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            color="#404040"
        )

        if prosent >= 18:
            ax.text(
                senter,
                prosent / 2,
                (
                    f"{poengtall(oppnådd)} av "
                    f"{poengtall(maksimum)} p"
                ),
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
                color="white",
                rotation=(
                    90
                    if len(etiketter) > 12
                    else 0
                )
            )

    ax.axhline(
        samlet_prosent,
        color=FARGE_TOTAL,
        linestyle="--",
        linewidth=2,
        zorder=2
    )

    ax.text(
        len(etiketter) - 0.55,
        samlet_prosent + 1.5,
        (
            "Samlet resultat: "
            f"{norsk_tall(samlet_prosent)} %"
        ),
        ha="right",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=FARGE_TOTAL
    )

    resultattekst = (
        f"Poengsum: "
        f"{poengtall(samlet_poeng)} av "
        f"{poengtall(samlet_maksimum)}\n"
        f"Prosent riktig: "
        f"{norsk_tall(samlet_prosent)} %\n"
        f"Avrundet prosent: "
        f"{avrundet_prosent} %\n"
        f"Karakter: {karakter}"
    )

    ax.text(
        0.99,
        0.97,
        resultattekst,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=10,
        color="#404040",
        bbox={
            "boxstyle": "round,pad=0.6",
            "facecolor": "white",
            "edgecolor": "#BFBFBF",
            "alpha": 0.95,
        }
    )

    forklaring = [
        Patch(
            facecolor=FARGE_LAV,
            label="Under 40 %"
        ),
        Patch(
            facecolor=FARGE_MIDDELS,
            label="40–59,9 %"
        ),
        Patch(
            facecolor=FARGE_GOD,
            label="60–79,9 %"
        ),
        Patch(
            facecolor=FARGE_MEGET_GOD,
            label="80–100 %"
        ),
    ]

    ax.legend(
        handles=forklaring,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=4,
        frameon=False,
        fontsize=9
    )

    fig.text(
        0.07,
        0.145,
        "Tilbakemelding:",
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="#303030"
    )

    fig.text(
        0.07,
        0.035,
        tilbakemelding_brytet,
        ha="left",
        va="bottom",
        fontsize=10,
        color="#404040",
        linespacing=1.35,
        bbox={
            "boxstyle": "round,pad=0.7",
            "facecolor": FARGE_TILBAKEMELDING,
            "edgecolor": FARGE_TILBAKEMELDING_KANT,
            "linewidth": 1.0,
        }
    )

    lagret_fil = None

    if lagre:
        RESULTATMAPPE.mkdir(
            parents=True,
            exist_ok=True
        )

        filformat = filformat.lower().strip()

        if filformat not in {
            "png",
            "pdf"
        }:
            raise ValueError(
                "Filformatet må være PNG eller PDF."
            )

        lagret_fil = RESULTATMAPPE / (
            f"{sikkert_filnavn(elevnavn)}"
            f"_resultat.{filformat}"
        )

        fig.savefig(
            lagret_fil,
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

    if vis:
        plt.show()

    else:
        plt.close(fig)

    return lagret_fil


# ============================================================
# 16. ELEVMENY OG SKJEMAFELT
# ============================================================

def oppdater_elevmeny(valgt_navn=None):
    """
    Oppdaterer listen over registrerte elever.
    """
    sorterte_navn = sorted(
        elevdata.keys(),
        key=str.casefold
    )

    elevvelger.options = sorterte_navn

    if valgt_navn in sorterte_navn:
        elevvelger.value = valgt_navn

    elif sorterte_navn:
        elevvelger.value = sorterte_navn[0]

    else:
        elevvelger.value = None


def tom_feltene(
    knapp=None,
    vis_melding=True
):
    """
    Gjør feltene klare for en ny elev.
    """
    navnefelt.value = ""

    for felt in poengfelter:
        felt.value = 0

    innlimingsfelt.value = ""
    kommentarfelt.value = ""

    if vis_melding:
        vis_status(
            "Feltene er tømt. Du kan registrere en ny elev.",
            "info"
        )


def last_elev_i_feltene(elevnavn):
    """
    Laster valgt elev inn i registreringsfeltene.
    """
    if elevnavn not in elevdata:
        return

    navnefelt.value = elevnavn

    for felt, verdi in zip(
        poengfelter,
        elevdata[elevnavn]["poeng"]
    ):
        felt.value = float(verdi)

    kommentarfelt.value = elevdata[
        elevnavn
    ].get(
        "kommentar",
        ""
    )

    innlimingsfelt.value = "\n".join(
        poengtall(verdi)
        for verdi in elevdata[elevnavn]["poeng"]
    )


# ============================================================
# 17. LEGG TIL, OPPDATER OG SLETT ELEV
# ============================================================

def legg_til_elev(knapp=None):
    """
    Legger til én elev manuelt.
    """
    try:
        (
            elevnavn,
            poengliste,
            kommentar
        ) = hent_inntasting()

        eksisterende_navn = finn_elev(
            elevnavn
        )

        if eksisterende_navn is not None:
            raise ValueError(
                f"{eksisterende_navn} finnes allerede. "
                "Bruk Oppdater elev."
            )

        elevdata[elevnavn] = {
            "poeng": poengliste,
            "kommentar": kommentar,
        }

        oppdater_elevmeny(
            valgt_navn=elevnavn
        )

        (
            _,
            samlet_poeng,
            samlet_maksimum,
            _,
            avrundet_prosent,
            karakter
        ) = beregn_resultat(
            elevnavn
        )

        vis_status(
            f"{elevnavn} er lagt til. "
            f"Resultat: {poengtall(samlet_poeng)} av "
            f"{poengtall(samlet_maksimum)} poeng, "
            f"{avrundet_prosent} %, karakter {karakter}.",
            "suksess"
        )

        vis_valgt_diagram()

    except Exception as feil:
        vis_status(
            f"Kunne ikke legge til elev: {feil}",
            "feil"
        )


def oppdater_elev(knapp=None):
    """
    Oppdaterer valgt elev.
    """
    try:
        gammel_elev = elevvelger.value

        if gammel_elev is None:
            raise ValueError(
                "Ingen elev er valgt."
            )

        (
            nytt_navn,
            nye_poeng,
            ny_kommentar
        ) = hent_inntasting()

        eksisterende_navn = finn_elev(
            nytt_navn
        )

        if (
            eksisterende_navn is not None
            and eksisterende_navn != gammel_elev
        ):
            raise ValueError(
                f"Navnet {eksisterende_navn} brukes allerede."
            )

        if nytt_navn != gammel_elev:
            del elevdata[gammel_elev]

        elevdata[nytt_navn] = {
            "poeng": nye_poeng,
            "kommentar": ny_kommentar,
        }

        oppdater_elevmeny(
            valgt_navn=nytt_navn
        )

        vis_status(
            f"{nytt_navn} er oppdatert.",
            "suksess"
        )

        vis_valgt_diagram()

    except Exception as feil:
        vis_status(
            f"Kunne ikke oppdatere elev: {feil}",
            "feil"
        )


def slett_elev(knapp=None):
    """
    Sletter valgt elev.
    """
    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    del elevdata[valgt_elev]

    oppdater_elevmeny()

    if elevvelger.value is not None:
        last_elev_i_feltene(
            elevvelger.value
        )

    else:
        tom_feltene(
            vis_melding=False
        )

    vis_status(
        f"{valgt_elev} er slettet.",
        "advarsel"
    )

    vis_valgt_diagram()


def slett_alle_elever(knapp=None):
    """
    Sletter alle registrerte elever.
    """
    if not elevdata:
        vis_status(
            "Elevlisten er allerede tom.",
            "info"
        )
        return

    if not bekreft_slett.value:
        vis_status(
            "Merk av Bekreft sletting av alle elever først.",
            "feil"
        )
        return

    antall = len(
        elevdata
    )

    elevdata.clear()

    bekreft_slett.value = False

    oppdater_elevmeny()

    tom_feltene(
        vis_melding=False
    )

    vis_status(
        f"{antall} elever er slettet.",
        "advarsel"
    )

    vis_valgt_diagram()


# ============================================================
# 18. VELG OG VIS ELEV
# ============================================================

def elev_valgt(endring):
    """
    Laster valgt elev.
    """
    elevnavn = endring["new"]

    if elevnavn is None:
        return

    last_elev_i_feltene(
        elevnavn
    )

    vis_valgt_diagram()


def vis_valgt_diagram(knapp=None):
    """
    Viser diagrammet for valgt elev.
    """
    with diagramomrade:
        clear_output(wait=True)

        valgt_elev = elevvelger.value

        if valgt_elev is None:
            display(
                widgets.HTML(
                    value=(
                        "<div style='"
                        "padding:30px;"
                        "text-align:center;"
                        "color:#6B7280;"
                        "border:1px dashed #BFC5CE;"
                        "border-radius:6px;"
                        "'>"
                        "<b>Ingen elever er registrert ennå.</b>"
                        "<br><br>"
                        "Lim inn hele klassen fra Excel eller "
                        "registrer én elev manuelt."
                        "</div>"
                    )
                )
            )
            return

        lag_diagram(
            valgt_elev,
            lagre=False,
            vis=True
        )


# ============================================================
# 19. LAGRE DIAGRAMMER
# ============================================================

def lagre_valgt_png(knapp=None):
    """
    Lagrer valgt elevdiagram som PNG.
    """
    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    fil = lag_diagram(
        valgt_elev,
        lagre=True,
        filformat="png",
        vis=False
    )

    vis_status(
        f"PNG-diagrammet er lagret som: {fil.resolve()}",
        "suksess"
    )


def lagre_valgt_pdf(knapp=None):
    """
    Lagrer valgt elevdiagram som PDF.
    """
    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    fil = lag_diagram(
        valgt_elev,
        lagre=True,
        filformat="pdf",
        vis=False
    )

    vis_status(
        f"PDF-diagrammet er lagret som: {fil.resolve()}",
        "suksess"
    )


def lagre_alle_diagrammer(knapp=None):
    """
    Lagrer ett PNG-diagram for hver registrerte elev.
    """
    if not elevdata:
        vis_status(
            "Det finnes ingen elever å lagre.",
            "feil"
        )
        return

    with meldingomrade:
        clear_output(wait=True)

        print(
            f"Lagrer diagrammer for "
            f"{len(elevdata)} elever ..."
        )
        print()

        for nummer, elevnavn in enumerate(
            sorted(
                elevdata.keys(),
                key=str.casefold
            ),
            start=1
        ):
            fil = lag_diagram(
                elevnavn,
                lagre=True,
                filformat="png",
                vis=False
            )

            print(
                f"{nummer} av {len(elevdata)}: "
                f"{fil.name}"
            )

        print()
        print("Diagrammene er lagret i:")
        print(RESULTATMAPPE.resolve())

    vis_status(
        f"Alle diagrammene er lagret i "
        f"{RESULTATMAPPE.resolve()}",
        "suksess"
    )


# ============================================================
# 20. LAGRE OG LAST ELEVDATA
# ============================================================

def lagre_elevdata(knapp=None):
    """
    Lagrer prøveoppsett og elevdata som JSON.
    """
    if not proveoppsett:
        vis_status(
            "Ingen prøve er opprettet.",
            "feil"
        )
        return

    data_som_lagres = {
        "proveoppsett": proveoppsett,
        "elever": elevdata,
    }

    with open(
        DATAFIL,
        "w",
        encoding="utf-8"
    ) as fil:
        json.dump(
            data_som_lagres,
            fil,
            ensure_ascii=False,
            indent=2
        )

    vis_status(
        f"Elevdataene er lagret som: {DATAFIL.resolve()}",
        "suksess"
    )


def last_elevdata(knapp=None):
    """
    Laster prøveoppsett og elevdata fra JSON.
    """
    global proveoppsett
    global poengfelter

    if not DATAFIL.exists():
        vis_status(
            f"Finner ikke filen: {DATAFIL.resolve()}",
            "feil"
        )
        return

    try:
        with open(
            DATAFIL,
            "r",
            encoding="utf-8"
        ) as fil:
            data = json.load(
                fil
            )

        lastet_oppsett = data.get(
            "proveoppsett",
            []
        )

        lastede_elever = data.get(
            "elever",
            {}
        )

        kontrollert_oppsett = []

        for nummer, element in enumerate(
            lastet_oppsett,
            start=1
        ):
            tema = str(
                element["tema"]
            ).strip()

            oppgave = str(
                element["oppgave"]
            ).strip()

            maksimum = float(
                element["maks"]
            )

            if not tema or not oppgave:
                raise ValueError(
                    f"Oppgave {nummer} har manglende informasjon."
                )

            if (
                not np.isfinite(maksimum)
                or maksimum <= 0
            ):
                raise ValueError(
                    f"Oppgave {nummer} har ugyldig makspoeng."
                )

            kontrollert_oppsett.append(
                {
                    "tema": tema,
                    "oppgave": oppgave,
                    "maks": maksimum,
                }
            )

        proveoppsett = kontrollert_oppsett

        kontrollert_elevdata = {}

        for elevnavn, elevinnhold in lastede_elever.items():
            poengliste = [
                float(verdi)
                for verdi in elevinnhold.get(
                    "poeng",
                    []
                )
            ]

            kommentar = str(
                elevinnhold.get(
                    "kommentar",
                    ""
                )
            )

            kontroller_poengliste(
                elevnavn,
                poengliste
            )

            kontrollert_elevdata[elevnavn] = {
                "poeng": poengliste,
                "kommentar": kommentar,
            }

        elevdata.clear()

        elevdata.update(
            kontrollert_elevdata
        )

        oppsett_felt.value = "\n".join(
            (
                f"{element['tema']};"
                f"{element['oppgave']};"
                f"{poengtall(element['maks'])}"
            )
            for element in proveoppsett
        )

        poengfelter = [
            lag_poengfelt(element)
            for element in proveoppsett
        ]

        poeng_rutenett.children = tuple(
            poengfelter
        )

        totalt_makspoeng = sum(
            element["maks"]
            for element in proveoppsett
        )

        proveinfo.value = (
            "<div style='"
            "padding:12px;"
            "margin:10px 0;"
            "background:#EEF5FF;"
            "border-left:5px solid #5B9BD5;"
            "border-radius:5px;"
            "'>"
            "<b>Aktiv prøve:</b> "
            f"{len(proveoppsett)} oppgaver, "
            f"totalt <b>{poengtall(totalt_makspoeng)} "
            "poeng</b>."
            "</div>"
        )

        registreringsdel.layout.display = ""

        oppdater_elevmeny()

        if elevvelger.value is not None:
            last_elev_i_feltene(
                elevvelger.value
            )

        vis_valgt_diagram()

        vis_status(
            f"{len(elevdata)} elever er lastet.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke laste elevdata: {feil}",
            "feil"
        )


# ============================================================
# 21. WIDGETER FOR PRØVEOPPSETT
# ============================================================

oppsett_felt = widgets.Textarea(
    value=(
        "Tema 1;Oppgave 1;4\n"
        "Tema 2;Oppgave 2;3\n"
        "Tema 3;Oppgave 3;2\n"
        "Tema 4;Oppgave 4;8\n"
        "Tema 5;Oppgave 5;4\n"
        "Tema 6;Oppgave 6;2\n"
        "Tema 7;Oppgave 7;6\n"
        "Tema 8;Oppgave 8;6"
    ),
    description="Oppsett:",
    style={
        "description_width": "90px"
    },
    layout=widgets.Layout(
        width="800px",
        height="230px"
    )
)

opprett_knapp = widgets.Button(
    description="Opprett / oppdater prøve",
    icon="check",
    button_style="primary",
    layout=widgets.Layout(
        width="240px"
    )
)

proveinfo = widgets.HTML()


# ============================================================
# 22. WIDGETER FOR MASSEIMPORT
# ============================================================

masseinnlimingsfelt = widgets.Textarea(
    value="",
    placeholder=(
        "Kopier hele Excel-området og lim det inn her.\n\n"
        "Første kolonne: Elevnavn\n"
        "Neste kolonner: Poeng på oppgave 1, 2, 3 osv.\n\n"
        "Ikke ta med karakterkolonnen."
    ),
    description="Hele klassen:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="95%",
        height="270px"
    )
)

forhandsvis_knapp = widgets.Button(
    description="Kontroller Excel-data",
    icon="check",
    button_style="info",
    layout=widgets.Layout(
        width="220px"
    )
)

importer_klasse_knapp = widgets.Button(
    description="Importer hele klassen",
    icon="table",
    button_style="success",
    layout=widgets.Layout(
        width="220px"
    )
)

importer_lagre_knapp = widgets.Button(
    description="Importer og lagre alle PNG",
    icon="folder",
    button_style="warning",
    layout=widgets.Layout(
        width="260px"
    )
)


# ============================================================
# 23. WIDGETER FOR ÉN ELEV
# ============================================================

navnefelt = widgets.Text(
    value="",
    placeholder="Skriv elevens navn",
    description="Elevnavn:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="650px"
    )
)

innlimingsfelt = widgets.Textarea(
    value="",
    placeholder=(
        "Lim inn poengene til én elev.\n"
        "Du kan lime inn én Excel-rad eller "
        "én verdi per linje."
    ),
    description="Lim inn poeng:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="760px",
        height="150px"
    )
)

fordel_poeng_knapp = widgets.Button(
    description="Fordel innlimte poeng",
    icon="paste",
    button_style="info",
    layout=widgets.Layout(
        width="220px"
    )
)

kommentarfelt = widgets.Textarea(
    value="",
    placeholder=(
        "Valgfri lærerkommentar."
    ),
    description="Kommentar:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="760px",
        height="90px"
    )
)

poeng_rutenett = widgets.GridBox(
    children=[],
    layout=widgets.Layout(
        grid_template_columns=(
            "repeat(2, minmax(420px, 1fr))"
        ),
        grid_gap="8px 15px",
        width="100%"
    )
)


# ============================================================
# 24. ELEVMENY OG VISNINGSOMRÅDER
# ============================================================

elevvelger = widgets.Dropdown(
    options=[],
    value=None,
    description="Registrerte:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="650px"
    )
)

statusfelt = widgets.HTML()

diagramomrade = widgets.Output()

meldingomrade = widgets.Output()


# ============================================================
# 25. KNAPPER
# ============================================================

legg_til_knapp = widgets.Button(
    description="Legg til elev",
    icon="plus",
    button_style="success",
    layout=widgets.Layout(
        width="170px"
    )
)

oppdater_knapp = widgets.Button(
    description="Oppdater elev",
    icon="edit",
    button_style="primary",
    layout=widgets.Layout(
        width="170px"
    )
)

slett_knapp = widgets.Button(
    description="Slett elev",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(
        width="160px"
    )
)

tom_knapp = widgets.Button(
    description="Ny elev / tøm",
    icon="eraser",
    layout=widgets.Layout(
        width="170px"
    )
)

vis_knapp = widgets.Button(
    description="Vis diagram",
    icon="bar-chart",
    button_style="primary",
    layout=widgets.Layout(
        width="170px"
    )
)

png_knapp = widgets.Button(
    description="Lagre PNG",
    icon="save",
    button_style="success",
    layout=widgets.Layout(
        width="160px"
    )
)

pdf_knapp = widgets.Button(
    description="Lagre PDF",
    icon="file-pdf",
    button_style="info",
    layout=widgets.Layout(
        width="160px"
    )
)

lagre_alle_knapp = widgets.Button(
    description="Lagre alle PNG",
    icon="folder",
    button_style="warning",
    layout=widgets.Layout(
        width="180px"
    )
)

lagre_data_knapp = widgets.Button(
    description="Lagre elevdata",
    icon="download",
    layout=widgets.Layout(
        width="180px"
    )
)

last_data_knapp = widgets.Button(
    description="Last elevdata",
    icon="upload",
    layout=widgets.Layout(
        width="180px"
    )
)

bekreft_slett = widgets.Checkbox(
    value=False,
    description="Bekreft sletting av alle elever",
    indent=False,
    layout=widgets.Layout(
        width="290px"
    )
)

slett_alle_knapp = widgets.Button(
    description="Slett alle elever",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(
        width="180px"
    )
)


# ============================================================
# 26. KOBLE KNAPPENE
# ============================================================

opprett_knapp.on_click(
    opprett_prove
)

forhandsvis_knapp.on_click(
    forhåndsvis_excelimport
)

importer_klasse_knapp.on_click(
    importer_hele_klassen
)

importer_lagre_knapp.on_click(
    importer_og_lagre_alle
)

fordel_poeng_knapp.on_click(
    fordel_innlimte_poeng
)

legg_til_knapp.on_click(
    legg_til_elev
)

oppdater_knapp.on_click(
    oppdater_elev
)

slett_knapp.on_click(
    slett_elev
)

tom_knapp.on_click(
    tom_feltene
)

vis_knapp.on_click(
    vis_valgt_diagram
)

png_knapp.on_click(
    lagre_valgt_png
)

pdf_knapp.on_click(
    lagre_valgt_pdf
)

lagre_alle_knapp.on_click(
    lagre_alle_diagrammer
)

lagre_data_knapp.on_click(
    lagre_elevdata
)

last_data_knapp.on_click(
    last_elevdata
)

slett_alle_knapp.on_click(
    slett_alle_elever
)

elevvelger.observe(
    elev_valgt,
    names="value"
)


# ============================================================
# 27. BYGG PROGRAMVINDUET
# ============================================================

masseimportknapper = widgets.HBox(
    children=[
        forhandsvis_knapp,
        importer_klasse_knapp,
        importer_lagre_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

registreringsknapper = widgets.HBox(
    children=[
        legg_til_knapp,
        oppdater_knapp,
        slett_knapp,
        tom_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

slett_alle_rad = widgets.HBox(
    children=[
        bekreft_slett,
        slett_alle_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        align_items="center",
        flex_flow="row wrap"
    )
)

diagramknapper = widgets.HBox(
    children=[
        vis_knapp,
        png_knapp,
        pdf_knapp,
        lagre_alle_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

dataknapper = widgets.HBox(
    children=[
        lagre_data_knapp,
        last_data_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

registreringsdel = widgets.VBox(
    children=[
        proveinfo,

        widgets.HTML(
            value=(
                "<h3 style='margin-bottom:6px;'>"
                "2. Lim inn hele klassen fra Excel"
                "</h3>"

                "<p style='color:#595959;'>"
                "Marker navnene og alle poengkolonnene i Excel. "
                "Første kolonne skal være elevnavn. De neste "
                "kolonnene skal være poengene i samme rekkefølge "
                "som oppgavene ovenfor."
                "</p>"

                "<p style='color:#595959;'>"
                "<b>Ikke ta med karakterkolonnen.</b>"
                "</p>"
            )
        ),

        masseinnlimingsfelt,
        masseimportknapper,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:20px 0;"
                "'>"

                "<h3 style='margin-bottom:6px;'>"
                "3. Registrer eller rediger én elev"
                "</h3>"
            )
        ),

        navnefelt,

        widgets.HTML(
            value=(
                "<p style='margin-bottom:5px;'>"
                "<b>Alternativ A:</b> Lim inn poengene "
                "til én elev fra Excel."
                "</p>"
            )
        ),

        innlimingsfelt,
        fordel_poeng_knapp,

        widgets.HTML(
            value=(
                "<p style='margin:14px 0 5px 0;'>"
                "<b>Alternativ B:</b> Skriv eller korriger "
                "poengene direkte i feltene."
                "</p>"
            )
        ),

        poeng_rutenett,
        kommentarfelt,
        registreringsknapper,
        slett_alle_rad,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"

                "<h3 style='margin-bottom:6px;'>"
                "4. Velg registrert elev"
                "</h3>"
            )
        ),

        elevvelger,
        diagramknapper,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"

                "<h3 style='margin-bottom:6px;'>"
                "5. Lagre eller hent elevdata"
                "</h3>"
            )
        ),

        dataknapper,
    ]
)

registreringsdel.layout.display = "none"


# ============================================================
# 28. BYGG HELE GRENSESNITTET
# ============================================================

grensesnitt = widgets.VBox(
    children=[
        widgets.HTML(
            value=(
                "<h2 style='margin-bottom:4px;'>"
                "El Capo 3.0: elevresultater, karakter "
                "og tilbakemelding"
                "</h2>"

                "<p style='margin-top:0;color:#595959;'>"
                "Skriv én linje per oppgave med formatet "
                "<b>Tema;Oppgavenummer;Makspoeng</b>."
                "</p>"

                "<p style='color:#595959;'>"
                "Eksempel: "
                "<b>Brøk;Oppgave 4;5</b>"
                "</p>"

                "<h3 style='margin-bottom:6px;'>"
                "1. Opprett prøve"
                "</h3>"
            )
        ),

        oppsett_felt,
        opprett_knapp,
        statusfelt,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"
            )
        ),

        registreringsdel,
    ],

    layout=widgets.Layout(
        border="1px solid #D9D9D9",
        padding="18px",
        width="100%"
    )
)


# ============================================================
# 29. START PROGRAMMET
# ============================================================

display(
    grensesnitt,
    diagramomrade,
    meldingomrade
)

Output()

Output()

In [ ]:
# ============================================================
# EL CAPO 2.0
# INTERAKTIV REGISTRERING, DIAGRAM, KARAKTER OG TILBAKEMELDING
#
# Python 3.14.8
# Jupyter Notebook / JupyterLab
#
# Funksjoner:
# - Valgfritt antall oppgaver
# - Tema, oppgavenummer og makspoeng
# - Manuell poengregistrering
# - Innliming av poeng fra Excel
# - Kontroll av makspoeng per oppgave
# - Kontroll av total poengsum
# - Automatisk prosent og karakter
# - Automatisk elevtilbakemelding
# - Valgfri lærerkommentar
# - Legg til, oppdater og slett elev
# - Lagre diagram som PNG eller PDF
# - Lagre og laste prøve/elevdata som JSON
# ============================================================


# ============================================================
# 1. IMPORTER PAKKER
# ============================================================

from pathlib import Path
import html
import json
import re
import textwrap

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch

import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# 2. DATA OG FILER
# ============================================================

# Prøveoppsettet får dette formatet:
#
# [
#     {
#         "tema": "Tallregning",
#         "oppgave": "Oppgave 1",
#         "maks": 4
#     }
# ]

proveoppsett = []

# Elevdata får dette formatet:
#
# {
#     "Elevnavn": {
#         "poeng": [3, 2, 4],
#         "kommentar": "Valgfri lærerkommentar"
#     }
# }

elevdata = {}

# Poengfeltene opprettes automatisk.
poengfelter = []

# Mappe for lagrede diagrammer.
RESULTATMAPPE = Path("Elevdiagrammer")

# Fil med prøveoppsett og elevdata.
DATAFIL = Path("elevresultater_el_capo_2.json")


# ============================================================
# 3. INNSTILLINGER FOR TILBAKEMELDING
# ============================================================

# Resultat under 60 % regnes som et øvingsområde.
GRENSE_OVING = 60

# Resultat fra og med 80 % regnes som god mestring.
GRENSE_STERK = 80

# Maksimalt antall områder som nevnes.
MAKS_OVINGSOMRADER = 3
MAKS_STERKE_OMRADER = 2


# ============================================================
# 4. FARGER
# ============================================================

FARGE_LAV = "#D9534F"
FARGE_MIDDELS = "#F0AD4E"
FARGE_GOD = "#5B9BD5"
FARGE_MEGET_GOD = "#2E8B57"

FARGE_TOTAL = "#7030A0"
FARGE_BAKGRUNN = "#FAFAFA"
FARGE_RUTENETT = "#D9D9D9"

FARGE_TILBAKEMELDING = "#F3F7FB"
FARGE_TILBAKEMELDING_KANT = "#AFC4D8"


# ============================================================
# 5. HJELPEFUNKSJONER
# ============================================================

def norsk_tall(verdi, desimaler=1):
    """
    Formaterer tall med norsk desimalkomma.
    """

    return f"{float(verdi):.{desimaler}f}".replace(".", ",")


def poengtall(verdi):
    """
    Viser hele tall uten desimal og desimaltall med komma.

    Eksempler:
        3.0 blir 3
        2.5 blir 2,5
    """

    verdi = float(verdi)

    if verdi.is_integer():
        return str(int(verdi))

    return norsk_tall(verdi, 1)


def sikkert_filnavn(tekst):
    """
    Lager et filnavn som fungerer i Windows.
    """

    tekst = str(tekst).strip()

    tekst = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        tekst
    )

    tekst = re.sub(
        r"\s+",
        "_",
        tekst
    )

    return tekst.strip("._") or "elev"


def velg_farge(prosent):
    """
    Velger søylefarge etter resultatet.
    """

    if prosent < 40:
        return FARGE_LAV

    if prosent < 60:
        return FARGE_MIDDELS

    if prosent < 80:
        return FARGE_GOD

    return FARGE_MEGET_GOD


def formater_oppramsing(navn):
    """
    Lager en naturlig norsk oppramsing.
    """

    navn = list(navn)

    if len(navn) == 0:
        return ""

    if len(navn) == 1:
        return navn[0]

    if len(navn) == 2:
        return f"{navn[0]} og {navn[1]}"

    return (
        f"{', '.join(navn[:-1])} "
        f"og {navn[-1]}"
    )


def finn_elev(navn):
    """
    Finner en elev uten å skille mellom
    store og små bokstaver.
    """

    sokenavn = navn.strip().casefold()

    for eksisterende_navn in elevdata:
        if eksisterende_navn.casefold() == sokenavn:
            return eksisterende_navn

    return None


def visningsnavn(element):
    """
    Lager etiketten som brukes i felt og diagram.

    Eksempel:
        Oppgave 1: Tallregning
    """

    return (
        f"{element['oppgave']}: "
        f"{element['tema']}"
    )


def vis_status(melding, meldingstype="info"):
    """
    Viser statusmelding i programvinduet.
    """

    stiler = {
        "info": {
            "bakgrunn": "#F3F4F6",
            "kant": "#6B7280",
        },
        "suksess": {
            "bakgrunn": "#ECF8EF",
            "kant": "#2E8B57",
        },
        "advarsel": {
            "bakgrunn": "#FFF4E5",
            "kant": "#F0AD4E",
        },
        "feil": {
            "bakgrunn": "#FFF1F1",
            "kant": "#D9534F",
        },
    }

    stil = stiler.get(
        meldingstype,
        stiler["info"]
    )

    sikker_melding = html.escape(
        str(melding)
    ).replace(
        "\n",
        "<br>"
    )

    statusfelt.value = (
        f"<div style='"
        f"padding:10px 14px;"
        f"margin-top:8px;"
        f"background:{stil['bakgrunn']};"
        f"border-left:5px solid {stil['kant']};"
        f"border-radius:5px;"
        f"color:#303030;"
        f"'>"
        f"{sikker_melding}"
        f"</div>"
    )


# ============================================================
# 6. KARAKTERBEREGNING
# ============================================================

def karakter_fra_prosent(prosent):
    """
    Regner ut karakter fra samlet prosent.

    Prosenten avrundes til nærmeste hele prosent
    før karakteren bestemmes.

    Eksempler:
        94,4 % blir 94 % og karakter 6-
        94,5 % blir 95 % og karakter 6
        28,6 % blir 29 % og karakter 2
    """

    avrundet_prosent = int(
        np.floor(float(prosent) + 0.5)
    )

    avrundet_prosent = max(
        0,
        min(
            100,
            avrundet_prosent
        )
    )

    karakterskala = [
        (95, "6"),
        (92, "6-"),
        (88, "5+"),
        (84, "5"),
        (80, "5-"),
        (75, "4+"),
        (65, "4"),
        (60, "4-"),
        (55, "3+"),
        (45, "3"),
        (40, "3-"),
        (35, "2+"),
        (29, "2"),
        (21, "2-"),
        (0, "1"),
    ]

    for nedre_grense, karakter in karakterskala:
        if avrundet_prosent >= nedre_grense:
            return karakter, avrundet_prosent

    return "1", avrundet_prosent


# ============================================================
# 7. LES PRØVEOPPSETT
# ============================================================

def les_proveoppsett():
    """
    Leser prøveoppsettet fra tekstfeltet.

    Hver linje skal ha formatet:
        Tema;Oppgavenummer;Makspoeng

    Eksempel:
        Tallregning;Oppgave 1;4
    """

    linjer = [
        linje.strip()
        for linje in oppsett_felt.value.splitlines()
        if linje.strip()
    ]

    if not linjer:
        raise ValueError(
            "Skriv minst én oppgave."
        )

    nytt_oppsett = []

    for linjenummer, linje in enumerate(
        linjer,
        start=1
    ):
        deler = [
            deltekst.strip()
            for deltekst in linje.split(";")
        ]

        if len(deler) != 3:
            raise ValueError(
                f"Linje {linjenummer} må ha formatet "
                "Tema;Oppgavenummer;Makspoeng."
            )

        tema, oppgavenummer, maks_tekst = deler

        if not tema:
            raise ValueError(
                f"Linje {linjenummer} mangler tema."
            )

        if not oppgavenummer:
            raise ValueError(
                f"Linje {linjenummer} mangler oppgavenummer."
            )

        try:
            makspoeng = float(
                maks_tekst.replace(",", ".")
            )

        except ValueError:
            raise ValueError(
                f"Ugyldig makspoeng på linje "
                f"{linjenummer}: {maks_tekst!r}."
            )

        if not np.isfinite(makspoeng):
            raise ValueError(
                f"Makspoeng på linje {linjenummer} "
                "er ikke et gyldig tall."
            )

        if makspoeng <= 0:
            raise ValueError(
                f"Makspoeng på linje {linjenummer} "
                "må være større enn 0."
            )

        nytt_oppsett.append(
            {
                "tema": tema,
                "oppgave": oppgavenummer,
                "maks": makspoeng,
            }
        )

    etiketter = [
        visningsnavn(element).casefold()
        for element in nytt_oppsett
    ]

    if len(set(etiketter)) != len(etiketter):
        raise ValueError(
            "Hver kombinasjon av oppgavenummer og tema "
            "må være unik."
        )

    return nytt_oppsett


# ============================================================
# 8. LAG POENGFELT
# ============================================================

def lag_poengfelt(element):
    """
    Lager ett fritt tallfelt.

    Feltet endrer ikke tallet automatisk.
    Verdien kontrolleres ved innliming, registrering
    og oppdatering.
    """

    etikett = visningsnavn(
        element
    )

    maksimum = element[
        "maks"
    ]

    return widgets.FloatText(
        value=0.0,
        step=0.5,
        description=f"{etikett}:",
        style={
            "description_width": "250px"
        },
        layout=widgets.Layout(
            width="420px"
        ),
        tooltip=(
            "Maksimal poengsum: "
            f"{poengtall(maksimum)}"
        )
    )


# ============================================================
# 9. OPPRETT PRØVEN
# ============================================================

def opprett_prove(knapp=None):
    """
    Oppretter prøve og poengfelt.
    """

    global proveoppsett
    global poengfelter

    try:
        nytt_oppsett = les_proveoppsett()

        if elevdata:
            raise ValueError(
                "Slett alle registrerte elever før "
                "prøveoppsettet endres."
            )

        proveoppsett = nytt_oppsett

        poengfelter = [
            lag_poengfelt(element)
            for element in proveoppsett
        ]

        poeng_rutenett.children = tuple(
            poengfelter
        )

        totalt_makspoeng = sum(
            element["maks"]
            for element in proveoppsett
        )

        proveinfo.value = (
            "<div style='"
            "padding:12px;"
            "margin:10px 0;"
            "background:#EEF5FF;"
            "border-left:5px solid #5B9BD5;"
            "border-radius:5px;"
            "'>"
            f"<b>Aktiv prøve:</b> "
            f"{len(proveoppsett)} oppgaver, "
            f"totalt <b>{poengtall(totalt_makspoeng)} "
            f"poeng</b>."
            "</div>"
        )

        registreringsdel.layout.display = ""

        tom_feltene()

        vis_status(
            "Prøveoppsettet er opprettet. "
            "Du kan nå registrere elever.",
            "suksess"
        )

        with diagramomrade:
            clear_output(
                wait=True
            )

            display(
                widgets.HTML(
                    value=(
                        "<div style='"
                        "padding:25px;"
                        "text-align:center;"
                        "color:#6B7280;"
                        "border:1px dashed #BFC5CE;"
                        "border-radius:6px;"
                        "'>"
                        "Prøven er klar. Registrer den "
                        "første eleven."
                        "</div>"
                    )
                )
            )

    except Exception as feil:
        vis_status(
            f"Kunne ikke opprette prøve: {feil}",
            "feil"
        )


# ============================================================
# 10. LES INNLIMTE POENG
# ============================================================

def les_innlimte_poeng():
    """
    Leser poeng fra innlimingsfeltet.

    Godtar:
    - én verdi per linje
    - én Excel-rad med tabulator mellom verdiene
    - semikolon mellom verdiene
    - norsk desimalkomma
    """

    tekst = innlimingsfelt.value.strip()

    if not tekst:
        raise ValueError(
            "Lim inn elevens poeng først."
        )

    # Excel bruker tabulator mellom kopierte celler.
    tekst = tekst.replace(
        "\t",
        "\n"
    )

    # Tillat også semikolon som skille.
    tekst = tekst.replace(
        ";",
        "\n"
    )

    verdier = []

    for deltekst in tekst.splitlines():
        deltekst = deltekst.strip()

        if not deltekst:
            continue

        # Fjern eventuelle HTML-linjeskift.
        deltekst = re.sub(
            r"<br\s*/?>",
            "",
            deltekst,
            flags=re.IGNORECASE
        )

        deltekst = (
            deltekst
            .replace(" ", "")
            .replace(",", ".")
        )

        if not deltekst:
            continue

        try:
            verdi = float(
                deltekst
            )

        except ValueError:
            raise ValueError(
                f"Kunne ikke tolke {deltekst!r} "
                "som et poengtall."
            )

        verdier.append(
            verdi
        )

    return verdier


# ============================================================
# 11. FORDEL INNLIMTE POENG
# ============================================================

def fordel_innlimte_poeng(knapp=None):
    """
    Fordeler innlimte poeng til de riktige feltene.
    """

    try:
        if not proveoppsett:
            raise ValueError(
                "Opprett prøveoppsettet først."
            )

        verdier = les_innlimte_poeng()

        forventet_antall = len(
            proveoppsett
        )

        if len(verdier) != forventet_antall:
            raise ValueError(
                f"Du limte inn {len(verdier)} poengverdier, "
                f"men prøven har {forventet_antall} oppgaver."
            )

        for element, verdi in zip(
            proveoppsett,
            verdier
        ):
            etikett = visningsnavn(
                element
            )

            maksimum = element[
                "maks"
            ]

            if not np.isfinite(verdi):
                raise ValueError(
                    f"{etikett}: Verdien er ikke et "
                    "gyldig tall."
                )

            if verdi < 0:
                raise ValueError(
                    f"{etikett}: Poeng kan ikke være negativt."
                )

            if verdi > maksimum:
                raise ValueError(
                    f"{etikett}: Du har limt inn "
                    f"{poengtall(verdi)} poeng, men "
                    f"maksimal poengsum er "
                    f"{poengtall(maksimum)}."
                )

        for felt, verdi in zip(
            poengfelter,
            verdier
        ):
            felt.value = verdi

        samlet_poeng = sum(
            verdier
        )

        samlet_maksimum = sum(
            element["maks"]
            for element in proveoppsett
        )

        vis_status(
            f"Poengene er fordelt til feltene. "
            f"Foreløpig poengsum: "
            f"{poengtall(samlet_poeng)} av "
            f"{poengtall(samlet_maksimum)}. "
            "Kontroller verdiene og trykk Legg til elev.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke fordele poengene: {feil}",
            "feil"
        )


# ============================================================
# 12. HENT OG KONTROLLER ELEVINNTAsting
# ============================================================

def hent_inntasting():
    """
    Henter og kontrollerer elevens poeng.

    Ingen oppgave kan overstige eget makspoeng.
    Samlet poengsum kan ikke overstige prøvens total.
    """

    if not proveoppsett:
        raise ValueError(
            "Opprett prøveoppsettet først."
        )

    elevnavn = navnefelt.value.strip()

    if not elevnavn:
        raise ValueError(
            "Skriv elevens navn."
        )

    poengliste = [
        float(felt.value)
        for felt in poengfelter
    ]

    if len(poengliste) != len(proveoppsett):
        raise ValueError(
            "Antall poengfelt stemmer ikke med "
            "antall oppgaver."
        )

    for element, oppnådd in zip(
        proveoppsett,
        poengliste
    ):
        etikett = visningsnavn(
            element
        )

        maksimum = element[
            "maks"
        ]

        if not np.isfinite(oppnådd):
            raise ValueError(
                f"{etikett} inneholder ikke "
                "et gyldig tall."
            )

        if oppnådd < 0:
            raise ValueError(
                f"{etikett} kan ikke ha negativ poengsum."
            )

        if oppnådd > maksimum:
            raise ValueError(
                f"{etikett}: Du har skrevet "
                f"{poengtall(oppnådd)} poeng, men "
                f"maksimal poengsum er "
                f"{poengtall(maksimum)}."
            )

    samlet_poeng = sum(
        poengliste
    )

    samlet_maksimum = sum(
        element["maks"]
        for element in proveoppsett
    )

    if samlet_poeng > samlet_maksimum:
        raise ValueError(
            f"Samlet poengsum er "
            f"{poengtall(samlet_poeng)}, men "
            f"prøven har maksimalt "
            f"{poengtall(samlet_maksimum)} poeng."
        )

    kommentar = kommentarfelt.value.strip()

    return (
        elevnavn,
        poengliste,
        kommentar
    )


# ============================================================
# 13. BEREGN RESULTAT
# ============================================================

def beregn_resultat(elevnavn):
    """
    Beregner prosent per oppgave, samlet prosent
    og karakter.
    """

    if elevnavn not in elevdata:
        raise KeyError(
            f"Fant ikke eleven {elevnavn!r}."
        )

    poengliste = elevdata[elevnavn][
        "poeng"
    ]

    resultater = []

    for element, oppnådd in zip(
        proveoppsett,
        poengliste
    ):
        maksimum = element[
            "maks"
        ]

        prosent = (
            100
            * float(oppnådd)
            / float(maksimum)
        )

        resultater.append(
            {
                "tema": element["tema"],
                "oppgave": element["oppgave"],
                "etikett": visningsnavn(element),
                "oppnådd": float(oppnådd),
                "maksimum": float(maksimum),
                "prosent": prosent,
            }
        )

    samlet_poeng = sum(
        poengliste
    )

    samlet_maksimum = sum(
        element["maks"]
        for element in proveoppsett
    )

    samlet_prosent = (
        100
        * samlet_poeng
        / samlet_maksimum
    )

    karakter, avrundet_prosent = (
        karakter_fra_prosent(
            samlet_prosent
        )
    )

    return (
        resultater,
        samlet_poeng,
        samlet_maksimum,
        samlet_prosent,
        avrundet_prosent,
        karakter
    )


# ============================================================
# 14. AUTOMATISK TILBAKEMELDING
# ============================================================

def lag_tilbakemelding(
    resultater,
    samlet_prosent,
    karakter,
    kommentar=""
):
    """
    Lager positiv og konkret elevtilbakemelding.
    """

    ovingsomrader = sorted(
        [
            resultat
            for resultat in resultater
            if resultat["prosent"] < GRENSE_OVING
        ],
        key=lambda resultat: (
            resultat["prosent"],
            resultat["etikett"]
        )
    )[:MAKS_OVINGSOMRADER]

    sterke_omrader = sorted(
        [
            resultat
            for resultat in resultater
            if resultat["prosent"] >= GRENSE_STERK
        ],
        key=lambda resultat: (
            -resultat["prosent"],
            resultat["etikett"]
        )
    )[:MAKS_STERKE_OMRADER]

    tekstdeler = [
        f"Samlet resultat gir karakter {karakter}."
    ]

    if samlet_prosent < 40:
        tekstdeler.append(
            "Du bør arbeide videre med flere deler av prøven. "
            "Ta ett område om gangen og øv på framgangsmåten "
            "før du prøver tilsvarende oppgaver på egen hånd."
        )

    elif samlet_prosent < 60:
        tekstdeler.append(
            "Du mestrer deler av prøven, men noen temaer "
            "eller oppgavetyper trenger mer øving."
        )

    elif samlet_prosent < 80:
        tekstdeler.append(
            "Du viser god forståelse på mange deler av prøven. "
            "Videre øving på noen utvalgte områder kan gjøre "
            "resultatet enda bedre."
        )

    else:
        tekstdeler.append(
            "Du viser svært god forståelse av innholdet i prøven. "
            "Fortsett å bruke de gode framgangsmåtene dine."
        )

    if ovingsomrader:
        ovingsnavn = [
            resultat["etikett"]
            for resultat in ovingsomrader
        ]

        tekstdeler.append(
            "Du bør særlig øve mer på "
            f"{formater_oppramsing(ovingsnavn)}."
        )

    if sterke_omrader:
        sterke_navn = [
            resultat["etikett"]
            for resultat in sterke_omrader
        ]

        tekstdeler.append(
            "Du har vist særlig god mestring på "
            f"{formater_oppramsing(sterke_navn)}."
        )

    kommentar = kommentar.strip()

    if kommentar:
        tekstdeler.append(
            f"Lærerens kommentar: {kommentar}"
        )

    return " ".join(
        tekstdeler
    )


# ============================================================
# 15. LAG DIAGRAM
# ============================================================

def lag_diagram(
    elevnavn,
    lagre=False,
    filformat="png",
    vis=True
):
    """
    Lager søylediagram, resultatboks, karakter
    og automatisk tilbakemelding.
    """

    (
        resultater,
        samlet_poeng,
        samlet_maksimum,
        samlet_prosent,
        avrundet_prosent,
        karakter
    ) = beregn_resultat(
        elevnavn
    )

    kommentar = elevdata[elevnavn].get(
        "kommentar",
        ""
    )

    tilbakemelding = lag_tilbakemelding(
        resultater,
        samlet_prosent,
        karakter,
        kommentar
    )

    tilbakemelding_brytet = textwrap.fill(
        tilbakemelding,
        width=145
    )

    etiketter = [
        resultat["etikett"]
        for resultat in resultater
    ]

    prosenter = np.array(
        [
            resultat["prosent"]
            for resultat in resultater
        ],
        dtype=float
    )

    # Automatisk bredde ved mange søyler.
    figurbredde = max(
        14,
        len(etiketter) * 1.15
    )

    fig, ax = plt.subplots(
        figsize=(
            figurbredde,
            9.8
        )
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.98,
        top=0.88,
        bottom=0.38
    )

    fig.patch.set_facecolor(
        "white"
    )

    ax.set_facecolor(
        FARGE_BAKGRUNN
    )

    x = np.arange(
        len(etiketter)
    )

    soyler = ax.bar(
        x,
        prosenter,
        width=0.70,
        color=[
            velg_farge(prosent)
            for prosent in prosenter
        ],
        edgecolor="white",
        linewidth=1.2,
        zorder=3
    )

    # Overskrift.
    ax.set_title(
        f"Resultat for: {elevnavn}",
        fontsize=18,
        fontweight="bold",
        color="#4F4F4F",
        pad=18
    )

    # Akser.
    ax.set_xlabel(
        "Tema / oppgave",
        fontsize=12,
        fontweight="bold",
        labelpad=14
    )

    ax.set_ylabel(
        "Prosent riktig",
        fontsize=12,
        fontweight="bold"
    )

    ax.set_ylim(
        0,
        110
    )

    ax.set_yticks(
        np.arange(
            0,
            101,
            10
        )
    )

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter(
            xmax=100,
            decimals=0
        )
    )

    ax.set_xticks(x)

    lengste_navn = max(
        len(navn)
        for navn in etiketter
    )

    if (
        len(etiketter) <= 7
        and lengste_navn <= 18
    ):
        rotasjon = 15
        skriftstorrelse = 10

    elif len(etiketter) <= 12:
        rotasjon = 30
        skriftstorrelse = 9

    else:
        rotasjon = 45
        skriftstorrelse = 8

    ax.set_xticklabels(
        etiketter,
        rotation=rotasjon,
        ha="right",
        fontsize=skriftstorrelse
    )

    # Rutenett.
    ax.grid(
        axis="y",
        color=FARGE_RUTENETT,
        linewidth=1,
        alpha=0.9,
        zorder=0
    )

    ax.grid(
        axis="x",
        visible=False
    )

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.spines["left"].set_color(
        "#BFBFBF"
    )

    ax.spines["bottom"].set_color(
        "#BFBFBF"
    )

    # Tekst ved søylene.
    for soyle, resultat in zip(
        soyler,
        resultater
    ):
        prosent = resultat[
            "prosent"
        ]

        oppnådd = resultat[
            "oppnådd"
        ]

        maksimum = resultat[
            "maksimum"
        ]

        senter = (
            soyle.get_x()
            + soyle.get_width() / 2
        )

        ax.text(
            senter,
            prosent + 1.5,
            f"{norsk_tall(prosent)} %",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            color="#404040"
        )

        if prosent >= 18:
            ax.text(
                senter,
                prosent / 2,
                (
                    f"{poengtall(oppnådd)} av "
                    f"{poengtall(maksimum)} p"
                ),
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
                color="white",
                rotation=(
                    90
                    if len(etiketter) > 12
                    else 0
                )
            )

    # Samlet prosentlinje.
    ax.axhline(
        samlet_prosent,
        color=FARGE_TOTAL,
        linestyle="--",
        linewidth=2,
        zorder=2
    )

    ax.text(
        len(etiketter) - 0.55,
        samlet_prosent + 1.5,
        (
            "Samlet resultat: "
            f"{norsk_tall(samlet_prosent)} %"
        ),
        ha="right",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=FARGE_TOTAL
    )

    # Resultatboks.
    resultattekst = (
        f"Poengsum: "
        f"{poengtall(samlet_poeng)} av "
        f"{poengtall(samlet_maksimum)}\n"
        f"Prosent riktig: "
        f"{norsk_tall(samlet_prosent)} %\n"
        f"Avrundet prosent: "
        f"{avrundet_prosent} %\n"
        f"Karakter: {karakter}"
    )

    ax.text(
        0.99,
        0.97,
        resultattekst,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=10,
        color="#404040",
        bbox={
            "boxstyle": "round,pad=0.6",
            "facecolor": "white",
            "edgecolor": "#BFBFBF",
            "alpha": 0.95,
        }
    )

    # Fargeforklaring.
    forklaring = [
        Patch(
            facecolor=FARGE_LAV,
            label="Under 40 %"
        ),
        Patch(
            facecolor=FARGE_MIDDELS,
            label="40–59,9 %"
        ),
        Patch(
            facecolor=FARGE_GOD,
            label="60–79,9 %"
        ),
        Patch(
            facecolor=FARGE_MEGET_GOD,
            label="80–100 %"
        ),
    ]

    ax.legend(
        handles=forklaring,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=4,
        frameon=False,
        fontsize=9
    )

    # Tilbakemelding.
    fig.text(
        0.07,
        0.145,
        "Tilbakemelding:",
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="#303030"
    )

    fig.text(
        0.07,
        0.035,
        tilbakemelding_brytet,
        ha="left",
        va="bottom",
        fontsize=10,
        color="#404040",
        linespacing=1.35,
        bbox={
            "boxstyle": "round,pad=0.7",
            "facecolor": FARGE_TILBAKEMELDING,
            "edgecolor": FARGE_TILBAKEMELDING_KANT,
            "linewidth": 1.0,
        }
    )

    # Lagring.
    lagret_fil = None

    if lagre:
        RESULTATMAPPE.mkdir(
            parents=True,
            exist_ok=True
        )

        filformat = filformat.lower().strip()

        if filformat not in {
            "png",
            "pdf"
        }:
            raise ValueError(
                "Filformatet må være PNG eller PDF."
            )

        lagret_fil = RESULTATMAPPE / (
            f"{sikkert_filnavn(elevnavn)}"
            f"_resultat.{filformat}"
        )

        fig.savefig(
            lagret_fil,
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

    if vis:
        plt.show()

    else:
        plt.close(fig)

    return lagret_fil


# ============================================================
# 16. OPPDATER ELEVMENY
# ============================================================

def oppdater_elevmeny(valgt_navn=None):
    """
    Oppdaterer listen over elever.
    """

    sorterte_navn = sorted(
        elevdata.keys(),
        key=str.casefold
    )

    elevvelger.options = sorterte_navn

    if valgt_navn in sorterte_navn:
        elevvelger.value = valgt_navn

    elif sorterte_navn:
        elevvelger.value = sorterte_navn[0]

    else:
        elevvelger.value = None


# ============================================================
# 17. TØM OG FYLL FELTENE
# ============================================================

def tom_feltene(knapp=None):
    """
    Gjør feltene klare for ny elev.
    """

    navnefelt.value = ""

    for felt in poengfelter:
        felt.value = 0

    innlimingsfelt.value = ""
    kommentarfelt.value = ""

    vis_status(
        "Feltene er tømt. Du kan registrere en ny elev.",
        "info"
    )


def last_elev_i_feltene(elevnavn):
    """
    Laster valgt elev inn i feltene.
    """

    if elevnavn not in elevdata:
        return

    navnefelt.value = elevnavn

    for felt, verdi in zip(
        poengfelter,
        elevdata[elevnavn]["poeng"]
    ):
        felt.value = float(
            verdi
        )

    kommentarfelt.value = elevdata[
        elevnavn
    ].get(
        "kommentar",
        ""
    )

    innlimingsfelt.value = "\n".join(
        norsk_tall(
            verdi,
            1
        )
        for verdi in elevdata[elevnavn]["poeng"]
    )


# ============================================================
# 18. LEGG TIL ELEV
# ============================================================

def legg_til_elev(knapp=None):
    """
    Legger til en ny elev.
    """

    try:
        (
            elevnavn,
            poengliste,
            kommentar
        ) = hent_inntasting()

        eksisterende_navn = finn_elev(
            elevnavn
        )

        if eksisterende_navn is not None:
            raise ValueError(
                f"{eksisterende_navn} finnes allerede. "
                "Bruk Oppdater elev for å endre resultatet."
            )

        elevdata[elevnavn] = {
            "poeng": poengliste,
            "kommentar": kommentar,
        }

        oppdater_elevmeny(
            valgt_navn=elevnavn
        )

        (
            _,
            samlet_poeng,
            samlet_maksimum,
            samlet_prosent,
            avrundet_prosent,
            karakter
        ) = beregn_resultat(
            elevnavn
        )

        vis_status(
            f"{elevnavn} er lagt til. "
            f"Resultat: {poengtall(samlet_poeng)} av "
            f"{poengtall(samlet_maksimum)} poeng, "
            f"{avrundet_prosent} %, karakter {karakter}.",
            "suksess"
        )

        vis_valgt_diagram()

    except Exception as feil:
        vis_status(
            f"Kunne ikke legge til elev: {feil}",
            "feil"
        )


# ============================================================
# 19. OPPDATER ELEV
# ============================================================

def oppdater_elev(knapp=None):
    """
    Oppdaterer valgt elev.
    """

    try:
        gammel_elev = elevvelger.value

        if gammel_elev is None:
            raise ValueError(
                "Ingen elev er valgt."
            )

        (
            nytt_navn,
            nye_poeng,
            ny_kommentar
        ) = hent_inntasting()

        eksisterende_navn = finn_elev(
            nytt_navn
        )

        if (
            eksisterende_navn is not None
            and eksisterende_navn != gammel_elev
        ):
            raise ValueError(
                f"Navnet {eksisterende_navn} brukes allerede."
            )

        if nytt_navn != gammel_elev:
            del elevdata[gammel_elev]

        elevdata[nytt_navn] = {
            "poeng": nye_poeng,
            "kommentar": ny_kommentar,
        }

        oppdater_elevmeny(
            valgt_navn=nytt_navn
        )

        (
            _,
            samlet_poeng,
            samlet_maksimum,
            samlet_prosent,
            avrundet_prosent,
            karakter
        ) = beregn_resultat(
            nytt_navn
        )

        vis_status(
            f"{nytt_navn} er oppdatert. "
            f"Resultat: {poengtall(samlet_poeng)} av "
            f"{poengtall(samlet_maksimum)} poeng, "
            f"{avrundet_prosent} %, karakter {karakter}.",
            "suksess"
        )

        vis_valgt_diagram()

    except Exception as feil:
        vis_status(
            f"Kunne ikke oppdatere elev: {feil}",
            "feil"
        )


# ============================================================
# 20. SLETT ELEV
# ============================================================

def slett_elev(knapp=None):
    """
    Sletter valgt elev.
    """

    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    del elevdata[valgt_elev]

    oppdater_elevmeny()

    if elevvelger.value is not None:
        last_elev_i_feltene(
            elevvelger.value
        )

    else:
        navnefelt.value = ""

        for felt in poengfelter:
            felt.value = 0

        innlimingsfelt.value = ""
        kommentarfelt.value = ""

    vis_status(
        f"{valgt_elev} er slettet.",
        "advarsel"
    )

    vis_valgt_diagram()


# ============================================================
# 21. SLETT ALLE ELEVER
# ============================================================

def slett_alle_elever(knapp=None):
    """
    Sletter alle elever etter bekreftelse.
    """

    if not elevdata:
        vis_status(
            "Elevlisten er allerede tom.",
            "info"
        )
        return

    if not bekreft_slett.value:
        vis_status(
            "Merk av Bekreft sletting av alle elever først.",
            "feil"
        )
        return

    antall = len(
        elevdata
    )

    elevdata.clear()

    bekreft_slett.value = False

    oppdater_elevmeny()

    tom_feltene()

    vis_status(
        f"{antall} elever er slettet.",
        "advarsel"
    )

    vis_valgt_diagram()


# ============================================================
# 22. VELG OG VIS ELEV
# ============================================================

def elev_valgt(endring):
    """
    Laster valgt elev.
    """

    elevnavn = endring["new"]

    if elevnavn is None:
        return

    last_elev_i_feltene(
        elevnavn
    )

    vis_valgt_diagram()


def vis_valgt_diagram(knapp=None):
    """
    Viser diagrammet for valgt elev.
    """

    with diagramomrade:
        clear_output(
            wait=True
        )

        valgt_elev = elevvelger.value

        if valgt_elev is None:
            display(
                widgets.HTML(
                    value=(
                        "<div style='"
                        "padding:30px;"
                        "text-align:center;"
                        "color:#6B7280;"
                        "border:1px dashed #BFC5CE;"
                        "border-radius:6px;"
                        "'>"
                        "<b>Ingen elever er registrert ennå.</b>"
                        "<br><br>"
                        "Skriv elevnavn og poeng, og trykk "
                        "<b>Legg til elev</b>."
                        "</div>"
                    )
                )
            )

            return

        lag_diagram(
            valgt_elev,
            lagre=False,
            vis=True
        )


# ============================================================
# 23. LAGRE DIAGRAMMER
# ============================================================

def lagre_valgt_png(knapp=None):
    """
    Lagrer valgt diagram som PNG.
    """

    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    fil = lag_diagram(
        valgt_elev,
        lagre=True,
        filformat="png",
        vis=False
    )

    vis_status(
        f"PNG-diagrammet er lagret som: {fil.resolve()}",
        "suksess"
    )


def lagre_valgt_pdf(knapp=None):
    """
    Lagrer valgt diagram som PDF.
    """

    valgt_elev = elevvelger.value

    if valgt_elev is None:
        vis_status(
            "Ingen elev er valgt.",
            "feil"
        )
        return

    fil = lag_diagram(
        valgt_elev,
        lagre=True,
        filformat="pdf",
        vis=False
    )

    vis_status(
        f"PDF-diagrammet er lagret som: {fil.resolve()}",
        "suksess"
    )


def lagre_alle_diagrammer(knapp=None):
    """
    Lagrer PNG-diagram for alle elever.
    """

    if not elevdata:
        vis_status(
            "Det finnes ingen elever å lagre.",
            "feil"
        )
        return

    with meldingomrade:
        clear_output(
            wait=True
        )

        print(
            f"Lagrer diagrammer for "
            f"{len(elevdata)} elever ..."
        )

        for nummer, elevnavn in enumerate(
            sorted(
                elevdata.keys(),
                key=str.casefold
            ),
            start=1
        ):
            fil = lag_diagram(
                elevnavn,
                lagre=True,
                filformat="png",
                vis=False
            )

            print(
                f"{nummer} av {len(elevdata)}: "
                f"{fil.name}"
            )

        print()
        print("Diagrammene er lagret i:")
        print(RESULTATMAPPE.resolve())

    vis_status(
        f"Alle diagrammene er lagret i "
        f"{RESULTATMAPPE.resolve()}",
        "suksess"
    )


# ============================================================
# 24. LAGRE ELEVDATA
# ============================================================

def lagre_elevdata(knapp=None):
    """
    Lagrer prøveoppsett og elevdata som JSON.
    """

    if not proveoppsett:
        vis_status(
            "Ingen prøve er opprettet.",
            "feil"
        )
        return

    data_som_lagres = {
        "proveoppsett": proveoppsett,
        "elever": elevdata,
    }

    with open(
        DATAFIL,
        "w",
        encoding="utf-8"
    ) as fil:
        json.dump(
            data_som_lagres,
            fil,
            ensure_ascii=False,
            indent=2
        )

    vis_status(
        f"Elevdataene er lagret som: {DATAFIL.resolve()}",
        "suksess"
    )


# ============================================================
# 25. LAST ELEVDATA
# ============================================================

def last_elevdata(knapp=None):
    """
    Laster prøveoppsett og elever fra JSON.
    """

    global proveoppsett
    global poengfelter

    if not DATAFIL.exists():
        vis_status(
            f"Finner ikke filen: {DATAFIL.resolve()}",
            "feil"
        )
        return

    try:
        with open(
            DATAFIL,
            "r",
            encoding="utf-8"
        ) as fil:
            data = json.load(
                fil
            )

        lastet_oppsett = data.get(
            "proveoppsett",
            []
        )

        lastede_elever = data.get(
            "elever",
            {}
        )

        kontrollert_oppsett = []

        for nummer, element in enumerate(
            lastet_oppsett,
            start=1
        ):
            tema = str(
                element["tema"]
            ).strip()

            oppgave = str(
                element["oppgave"]
            ).strip()

            maksimum = float(
                element["maks"]
            )

            if not tema:
                raise ValueError(
                    f"Oppgave {nummer} mangler tema."
                )

            if not oppgave:
                raise ValueError(
                    f"Oppgave {nummer} mangler oppgavenummer."
                )

            if (
                not np.isfinite(maksimum)
                or maksimum <= 0
            ):
                raise ValueError(
                    f"Oppgave {nummer} har ugyldig makspoeng."
                )

            kontrollert_oppsett.append(
                {
                    "tema": tema,
                    "oppgave": oppgave,
                    "maks": maksimum,
                }
            )

        kontrollert_elevdata = {}

        total_maksimum = sum(
            element["maks"]
            for element in kontrollert_oppsett
        )

        for elevnavn, elevinnhold in lastede_elever.items():
            poengliste = [
                float(verdi)
                for verdi in elevinnhold.get(
                    "poeng",
                    []
                )
            ]

            kommentar = str(
                elevinnhold.get(
                    "kommentar",
                    ""
                )
            )

            if (
                len(poengliste)
                != len(kontrollert_oppsett)
            ):
                raise ValueError(
                    f"{elevnavn} har feil antall poengverdier."
                )

            for element, oppnådd in zip(
                kontrollert_oppsett,
                poengliste
            ):
                if (
                    not np.isfinite(oppnådd)
                    or oppnådd < 0
                    or oppnådd > element["maks"]
                ):
                    raise ValueError(
                        f"{elevnavn} har ugyldig poengsum på "
                        f"{visningsnavn(element)}."
                    )

            if sum(poengliste) > total_maksimum:
                raise ValueError(
                    f"{elevnavn} overstiger total makspoeng."
                )

            kontrollert_elevdata[elevnavn] = {
                "poeng": poengliste,
                "kommentar": kommentar,
            }

        proveoppsett = kontrollert_oppsett

        elevdata.clear()
        elevdata.update(
            kontrollert_elevdata
        )

        oppsett_felt.value = "\n".join(
            (
                f"{element['tema']};"
                f"{element['oppgave']};"
                f"{poengtall(element['maks'])}"
            )
            for element in proveoppsett
        )

        poengfelter = [
            lag_poengfelt(element)
            for element in proveoppsett
        ]

        poeng_rutenett.children = tuple(
            poengfelter
        )

        proveinfo.value = (
            "<div style='"
            "padding:12px;"
            "margin:10px 0;"
            "background:#EEF5FF;"
            "border-left:5px solid #5B9BD5;"
            "border-radius:5px;"
            "'>"
            f"<b>Aktiv prøve:</b> "
            f"{len(proveoppsett)} oppgaver, "
            f"totalt <b>{poengtall(total_maksimum)} "
            f"poeng</b>."
            "</div>"
        )

        registreringsdel.layout.display = ""

        oppdater_elevmeny()

        if elevvelger.value is not None:
            last_elev_i_feltene(
                elevvelger.value
            )

        vis_valgt_diagram()

        vis_status(
            f"{len(elevdata)} elever er lastet.",
            "suksess"
        )

    except Exception as feil:
        vis_status(
            f"Kunne ikke laste elevdata: {feil}",
            "feil"
        )


# ============================================================
# 26. WIDGETER FOR PRØVEOPPSETT
# ============================================================

oppsett_felt = widgets.Textarea(
    value=(
        "Tallregning;Oppgave 1;4\n"
        "Negative tall;Oppgave 2;3\n"
        "Regnerekkefølge;Oppgave 3;3\n"
        "Brøk;Oppgave 4;5\n"
        "Desimaltall;Oppgave 5;3\n"
        "Prosent;Oppgave 6;4\n"
        "Prosentvis endring;Oppgave 7;4\n"
        "Likninger;Oppgave 8;5\n"
        "Formler;Oppgave 9;3\n"
        "Proporsjonalitet;Oppgave 10;4\n"
        "Lineære funksjoner;Oppgave 11;6\n"
        "Geometri;Oppgave 12;4\n"
        "Areal og omkrets;Oppgave 13;4\n"
        "Statistikk;Oppgave 14;5\n"
        "Problemløsing;Oppgave 15;8"
    ),
    description="Oppsett:",
    style={
        "description_width": "90px"
    },
    layout=widgets.Layout(
        width="800px",
        height="280px"
    )
)

opprett_knapp = widgets.Button(
    description="Opprett / oppdater prøve",
    icon="check",
    button_style="primary",
    layout=widgets.Layout(
        width="240px"
    )
)

proveinfo = widgets.HTML()


# ============================================================
# 27. WIDGETER FOR ELEVINNTAsting
# ============================================================

navnefelt = widgets.Text(
    value="",
    placeholder="Skriv elevens navn",
    description="Elevnavn:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="650px"
    )
)

innlimingsfelt = widgets.Textarea(
    value="",
    placeholder=(
        "Lim inn én verdi per linje eller en rad fra Excel.\n"
        "Eksempel:\n"
        "0,0\n"
        "0,0\n"
        "1,0\n"
        "0,0\n"
        "3,0\n"
        "0,0\n"
        "0,0"
    ),
    description="Lim inn poeng:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="760px",
        height="170px"
    )
)

fordel_poeng_knapp = widgets.Button(
    description="Fordel innlimte poeng",
    icon="paste",
    button_style="info",
    layout=widgets.Layout(
        width="220px"
    )
)

kommentarfelt = widgets.Textarea(
    value="",
    placeholder=(
        "Valgfri lærerkommentar. Kommentaren vises "
        "etter den automatiske tilbakemeldingen."
    ),
    description="Kommentar:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="760px",
        height="90px"
    )
)

poeng_rutenett = widgets.GridBox(
    children=[],
    layout=widgets.Layout(
        grid_template_columns=(
            "repeat(2, minmax(420px, 1fr))"
        ),
        grid_gap="8px 15px",
        width="100%"
    )
)


# ============================================================
# 28. ELEVMENY OG VISNINGSOMRÅDER
# ============================================================

elevvelger = widgets.Dropdown(
    options=[],
    value=None,
    description="Registrerte:",
    style={
        "description_width": "120px"
    },
    layout=widgets.Layout(
        width="650px"
    )
)

statusfelt = widgets.HTML()

diagramomrade = widgets.Output()

meldingomrade = widgets.Output()


# ============================================================
# 29. KNAPPER
# ============================================================

legg_til_knapp = widgets.Button(
    description="Legg til elev",
    icon="plus",
    button_style="success",
    layout=widgets.Layout(
        width="170px"
    )
)

oppdater_knapp = widgets.Button(
    description="Oppdater elev",
    icon="edit",
    button_style="primary",
    layout=widgets.Layout(
        width="170px"
    )
)

slett_knapp = widgets.Button(
    description="Slett elev",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(
        width="160px"
    )
)

tom_knapp = widgets.Button(
    description="Ny elev / tøm",
    icon="eraser",
    layout=widgets.Layout(
        width="170px"
    )
)

vis_knapp = widgets.Button(
    description="Vis diagram",
    icon="bar-chart",
    button_style="primary",
    layout=widgets.Layout(
        width="170px"
    )
)

png_knapp = widgets.Button(
    description="Lagre PNG",
    icon="save",
    button_style="success",
    layout=widgets.Layout(
        width="160px"
    )
)

pdf_knapp = widgets.Button(
    description="Lagre PDF",
    icon="file-pdf",
    button_style="info",
    layout=widgets.Layout(
        width="160px"
    )
)

lagre_alle_knapp = widgets.Button(
    description="Lagre alle PNG",
    icon="folder",
    button_style="warning",
    layout=widgets.Layout(
        width="180px"
    )
)

lagre_data_knapp = widgets.Button(
    description="Lagre elevdata",
    icon="download",
    layout=widgets.Layout(
        width="180px"
    )
)

last_data_knapp = widgets.Button(
    description="Last elevdata",
    icon="upload",
    layout=widgets.Layout(
        width="180px"
    )
)

bekreft_slett = widgets.Checkbox(
    value=False,
    description="Bekreft sletting av alle elever",
    indent=False,
    layout=widgets.Layout(
        width="290px"
    )
)

slett_alle_knapp = widgets.Button(
    description="Slett alle elever",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(
        width="180px"
    )
)


# ============================================================
# 30. KOBLE KNAPPENE
# ============================================================

opprett_knapp.on_click(
    opprett_prove
)

fordel_poeng_knapp.on_click(
    fordel_innlimte_poeng
)

legg_til_knapp.on_click(
    legg_til_elev
)

oppdater_knapp.on_click(
    oppdater_elev
)

slett_knapp.on_click(
    slett_elev
)

tom_knapp.on_click(
    tom_feltene
)

vis_knapp.on_click(
    vis_valgt_diagram
)

png_knapp.on_click(
    lagre_valgt_png
)

pdf_knapp.on_click(
    lagre_valgt_pdf
)

lagre_alle_knapp.on_click(
    lagre_alle_diagrammer
)

lagre_data_knapp.on_click(
    lagre_elevdata
)

last_data_knapp.on_click(
    last_elevdata
)

slett_alle_knapp.on_click(
    slett_alle_elever
)

elevvelger.observe(
    elev_valgt,
    names="value"
)


# ============================================================
# 31. BYGG REGISTRERINGSVINDUET
# ============================================================

registreringsknapper = widgets.HBox(
    children=[
        legg_til_knapp,
        oppdater_knapp,
        slett_knapp,
        tom_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

slett_alle_rad = widgets.HBox(
    children=[
        bekreft_slett,
        slett_alle_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        align_items="center",
        flex_flow="row wrap"
    )
)

diagramknapper = widgets.HBox(
    children=[
        vis_knapp,
        png_knapp,
        pdf_knapp,
        lagre_alle_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

dataknapper = widgets.HBox(
    children=[
        lagre_data_knapp,
        last_data_knapp,
    ],
    layout=widgets.Layout(
        gap="10px",
        flex_flow="row wrap"
    )
)

registreringsdel = widgets.VBox(
    children=[
        proveinfo,

        widgets.HTML(
            value=(
                "<h3 style='margin-bottom:6px;'>"
                "2. Registrer eller rediger elev"
                "</h3>"
            )
        ),

        navnefelt,

        widgets.HTML(
            value=(
                "<p style='margin-bottom:5px;'>"
                "<b>Alternativ A:</b> Lim inn alle poengene "
                "fra Excel."
                "</p>"
            )
        ),

        innlimingsfelt,
        fordel_poeng_knapp,

        widgets.HTML(
            value=(
                "<p style='margin:14px 0 5px 0;'>"
                "<b>Alternativ B:</b> Skriv eller korriger "
                "poengene direkte i feltene."
                "</p>"
            )
        ),

        poeng_rutenett,
        kommentarfelt,
        registreringsknapper,
        slett_alle_rad,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"
                "<h3 style='margin-bottom:6px;'>"
                "3. Velg registrert elev"
                "</h3>"
            )
        ),

        elevvelger,
        diagramknapper,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"
                "<h3 style='margin-bottom:6px;'>"
                "4. Lagre eller hent elevdata"
                "</h3>"
            )
        ),

        dataknapper,
    ]
)

# Skjules til prøven er opprettet.
registreringsdel.layout.display = "none"


# ============================================================
# 32. BYGG HELE PROGRAMVINDUET
# ============================================================

grensesnitt = widgets.VBox(
    children=[
        widgets.HTML(
            value=(
                "<h2 style='margin-bottom:4px;'>"
                "El Capo 2.0: elevresultater, karakter "
                "og tilbakemelding"
                "</h2>"

                "<p style='margin-top:0;color:#595959;'>"
                "Skriv én linje per oppgave med formatet "
                "<b>Tema;Oppgavenummer;Makspoeng</b>."
                "</p>"

                "<p style='color:#595959;'>"
                "Eksempel: "
                "<b>Brøk;Oppgave 4;5</b>"
                "</p>"

                "<h3 style='margin-bottom:6px;'>"
                "1. Opprett prøve"
                "</h3>"
            )
        ),

        oppsett_felt,
        opprett_knapp,
        statusfelt,

        widgets.HTML(
            value=(
                "<hr style='"
                "border:none;"
                "border-top:1px solid #D9D9D9;"
                "margin:18px 0;"
                "'>"
            )
        ),

        registreringsdel,
    ],

    layout=widgets.Layout(
        border="1px solid #D9D9D9",
        padding="18px",
        width="100%"
    )
)


# ============================================================
# 33. START PROGRAMMET
# ============================================================

display(
    grensesnitt,
    diagramomrade,
    meldingomrade
)